# functions

In [1]:
import pandas as pd
import glob
import os
# Set option to display all columns
pd.set_option('display.max_columns', None)


In [2]:
import pandas as pd
import numpy as np
from urllib.parse import urlparse
import re
import unicodedata

DOI_CORE_RE = re.compile(r"(10\.\d{4,9}/\S+)", re.IGNORECASE)

def _normalize_doi_raw(x: str) -> str | None:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    s = unicodedata.normalize("NFKC", s).lower()
    m = DOI_CORE_RE.search(s)  # extract from URL/“doi:” etc.
    if m:
        return m.group(1)
    if s.startswith("10.") and "/" in s:
        return s
    return None

def clean_preprint_fields(df: pd.DataFrame, *, numeric_keep: int = 2, add_bucket: bool = True) -> pd.DataFrame:
    """Clean + enrich preprint fields.
    - Builds doi_prefix_first_token
    - If first token is purely numeric, keeps only its first `numeric_keep` digits
    - Optionally adds a bucketing column for grouping
    """
    df = df.drop_duplicates().copy()

    # # --- gold_server_name
    # df["gold_server_name"] = (
    #     df.get("institution_name")
    #       .fillna(df.get("group_title"))
    #       .fillna(df.get("publisher"))
    # )

    # --- Normalize DOI
    if "doi" in df.columns:
        df["doi_lc"] = df["doi"].map(_normalize_doi_raw)
    elif "parent_doi" in df.columns:
        df["doi_lc"] = df["parent_doi"].map(_normalize_doi_raw)
    else:
        df["doi_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- prefix as given
    if "prefix" in df.columns:
        df["prefix_lc"] = df["prefix"].astype(str).str.strip().str.lower()
        df.loc[df["prefix_lc"].isin(["", "nan", "none"]), "prefix_lc"] = pd.NA
    else:
        df["prefix_lc"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Extract prefix/suffix from normalized DOI
    doi_parts = df["doi_lc"].str.extract(r"^(10\.\d{4,9})/(.+)$")
    df["doi_prefix_from_text"] = doi_parts[0]
    df["doi_suffix"] = doi_parts[1]
    df["prefix_lc"] = df["prefix_lc"].where(df["prefix_lc"].notna(), df["doi_prefix_from_text"])

    # --- Build first segment of suffix
    starts_with_letter = df["doi_suffix"].str.match(r"^[a-z]", na=False)

    # letters-case: take only leading letters/hyphens; stop before digits or separators
    first_seg_letters = df["doi_suffix"].str.extract(
        r"^([a-z\-]+)(?=\d|[.\-_/:]|$)", expand=False
    )

    # default-case: first chunk before separators (keeps digits)
    first_seg_default = df["doi_suffix"].str.split(r"[.\-_/:\s]", n=1, regex=True).str[0]

    first_seg = pd.Series(
        np.where(starts_with_letter, first_seg_letters, first_seg_default),
        index=df.index,
        dtype="object"
    )

    # fallback: permissive token if still NA
    need_fallback = first_seg.isna() & df["doi_suffix"].notna()
    first_seg.loc[need_fallback] = df.loc[need_fallback, "doi_suffix"].str.extract(r"^([a-z0-9\-]+)", expand=False)

    # --- NEW: compress purely numeric first tokens to first `numeric_keep` digits
    if numeric_keep and numeric_keep > 0:
        numeric_only = first_seg.str.fullmatch(r"\d+", na=False)
        first_seg.loc[numeric_only] = first_seg.loc[numeric_only].str[:numeric_keep]

    # --- Assemble final token
    df["doi_prefix_first_token"] = pd.Series(pd.NA, index=df.index, dtype="object")
    ok = df["prefix_lc"].notna() & first_seg.notna() & (first_seg.astype(str) != "")
    df.loc[ok, "doi_prefix_first_token"] = df.loc[ok, "prefix_lc"].astype(str) + "/" + first_seg.loc[ok].astype(str)

    # --- Optional bucket for grouping/plots (mirrors the compressed numeric rule)
    if add_bucket:
        df["doi_prefix_bucket_2d"] = df["doi_prefix_first_token"]

    # --- Domains
    def domain_and_first_path(u):
        try:
            parsed = urlparse(str(u).lower())
            host = parsed.netloc
            if host.startswith("www."):
                host = host[4:]
            parts = re.split(r"[/=]", parsed.path)
            first_part = parts[1] if len(parts) > 1 and parts[1] else None
            return f"{host}/{first_part}" if host and first_part else (host or None)
        except Exception:
            return None

    if "landing_page_url" in df.columns:
        df["primary_domain"] = df["landing_page_url"].apply(lambda u: urlparse(str(u)).netloc.lower().replace("www.", "") if pd.notna(u) else None)
        df["primary_domain_extend"] = df["landing_page_url"].apply(domain_and_first_path)
    elif "parent_url" in df.columns:
        df["primary_domain"] = df["parent_url"].apply(lambda u: urlparse(str(u)).netloc.lower().replace("www.", "") if pd.notna(u) else None)
        df["primary_domain_extend"] = df["parent_url"].apply(domain_and_first_path)
    else:
        df["primary_domain"] = pd.Series(pd.NA, index=df.index, dtype="object")
        df["primary_domain_extend"] = pd.Series(pd.NA, index=df.index, dtype="object")

    # --- Dates → year
    if "posted_date" in df.columns:
        df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce")
        df["year"] = df["posted_date"].dt.year

    return df

# # ----- usage
# df = clean_preprint_fields(df, numeric_keep=2, add_bucket=True)
# print(df.shape)

In [3]:
import pandas as pd
import glob
import os

def get_server_data(server_name, base_path=r"/mnt/c/SCHOLCOMMLAB/APPs/preprint-harvester/data/by_server/"):
    """
    Loads, cleans, and summarizes server metadata with clear visual formatting.
    """
    
    # 1. CONSTRUCTION & LOADING
    folder_path = os.path.join(base_path, server_name)
    parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))
    
    if not parquet_files:
        print(f"\n[!] ERROR: No parquet files found for '{server_name}'")
        print(f"    Path searched: {folder_path}\n")
        return None, None

    df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    df = df.drop_duplicates('record_id')
    
    # 2. STATUS HEADER
    print("\n" + "="*50)
    print(f" SERVER ANALYSIS: {server_name.upper()}")
    print("="*50)
    print(f"  > Files found:    {len(parquet_files)}")
    print(f"  > Raw records:    {len(df)}")
    summary = print('OK')
    
    return df, summary

# ----- Execution example
# elife_df = get_server_data("eLife")

In [4]:
import pandas as pd
import os

def analyze_parent_data(server_name, parent_df):
    """
    Filters the consolidated parent DataFrame for a specific server 
    and provides a summary of versioning and metadata.
    """
    
    # 1. FILTERING
    # Filter first to minimize memory usage during cleaning
    df_filtered = parent_df[parent_df['parent_server_name'] == server_name].copy()
    
    # Check if empty BEFORE running cleaning logic
    if df_filtered.empty:
        print(f"\n" + "!"*60)
        print(f" [!] ERROR: No records found for '{server_name}'")
        print(f" Check if the name matches 'parent_server_name' exactly.")
        print("!"*60 + "\n")
        return None, None

    # 2. CLEANING
    # Running cleaning on the filtered subset
    df = clean_preprint_fields(df_filtered, numeric_keep=2, add_bucket=True)
    
    # 3. STATUS HEADER
    print("\n" + "="*65)
    print(f" CONSOLIDATED ANALYSIS: {server_name.upper()}")
    print("="*65)
    print(f"  > Total Parent Groups:     {len(df):,}")
    print(f"  > Total Versions Found:    {int(df['total_versions'].sum()):,}")
    print(f"  > Unique Parent DOIs:      {df['parent_doi'].nunique():,}")
    print("-" * 65)

    # 4. VERSIONING SUMMARY
    print(f"\n VERSIONING DISTRIBUTION for {server_name}:")
    print("-" * 35)
    v_counts = df['total_versions'].value_counts().sort_index()
    for version_num, count in v_counts.items():
        print(f"  {int(version_num)} version(s):".ljust(20) + f"{count:,} groups")
    
    print(f"\n  Average versions per parent: {df['total_versions'].mean():.2f}")

    # 5. CROSS-SERVER MAPPING
    print(f"\n MOST RECENT DESTINATIONS (Migration):")
    print("-" * 35)
    destinations = df['most_recent_version_servers'].value_counts().head(10)
    print(destinations.to_string())

    # 6. METADATA SUMMARY
    cols_to_summarize = [
        "doi_prefix_first_token", "primary_domain"] # "servers_with_counts", "parent_year_first_seen", 
    summary = {}
    
    for col in cols_to_summarize:
        if col in df.columns:
            summary[col] = df[col].value_counts(dropna=False)
            print(f"\n TOP VALUES FOR: {col.upper()}")
            print("-" * 35)
            print(summary[col].head(10).to_string())
            print("\n")

    print("="*65)
    print(f" COMPLETED: {server_name.upper()}")
    print("="*65 + "\n")

    return df, summary

# ----- Execution Example
# parent_path = "outputs_new/parent/parent_version_index_full.parquet"
# full_parent_df = pd.read_parquet(parent_path)

# elife_parent_df, elife_parent_summary = analyze_parent_data("eLife", full_parent_df)

# Load parent data

In [5]:
# ----- Execution Example
# 1. Load the big file once
long_path = "outputs_new/parent/dedupe_clusters_long_full.parquet"
parent_path = "outputs_new/parent/parent_version_index_full.parquet"

full_parent_df = pd.read_parquet(parent_path)
# full_dataset_df = pd.read_parquet(long_path)
# 2. Analyze a specific server within that file
# arxiv_parent_df, arxiv_parent_summary = analyze_parent_data("arXiv", full_parent_df)
# advance_parent_df, advance_parent_summary = analyze_parent_data("Advance", full_parent_df)

# Advance

In [6]:
Advance_df, Advance_summary = get_server_data("Advance")


 SERVER ANALYSIS: ADVANCE
  > Files found:    1
  > Raw records:    4401
OK


In [7]:
Advance_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht..."
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht..."
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licens

In [8]:
Advance_df.columns

Index(['record_id', 'server_name', 'backend', 'source_work_id', 'doi',
       'doi_url', 'landing_page_url', 'url_best', 'prefix', 'member_id',
       'client_id', 'provider_id', 'source_registry', 'publisher',
       'container_title', 'institution_name', 'group_title', 'issn', 'title',
       'original_title', 'short_title', 'subtitle', 'language',
       'type_backend_raw', 'subtype_backend_raw', 'type_canonical',
       'is_paratext', 'is_preprint_candidate', 'date_created', 'date_posted',
       'date_deposited', 'date_indexed', 'date_updated', 'date_issued',
       'date_registered', 'date_published', 'date_published_online',
       'publication_year', 'date_published_source', 'date_posted_source',
       'is_oa', 'oa_status', 'license', 'license_url_best', 'abstract_raw',
       'abstract_text', 'links_json_best', 'fulltext_pdf_url', 'authors_flat',
       'institutions_flat', 'countries_flat', 'authors_json',
       'contributors_json', 'editors_json', 'funders_json', 'funders_

In [9]:
df=Advance_df.copy()

## observation

it seems like their create the new doi with versions patterns and the old records without version patterns are not available on the main page

In [10]:
df['doi'].sort_values().head(60)#.tolist()


354        10.31124/advance.10005662
1779    10.31124/advance.10005662.v1
1780    10.31124/advance.10005662.v2
340        10.31124/advance.10005884
1781    10.31124/advance.10005884.v1
346        10.31124/advance.10007411
1782    10.31124/advance.10007411.v1
342        10.31124/advance.10010381
1783    10.31124/advance.10010381.v1
343        10.31124/advance.10012031
1785    10.31124/advance.10012031.v1
355        10.31124/advance.10026860
1784    10.31124/advance.10026860.v1
390        10.31124/advance.10048160
1768    10.31124/advance.10048160.v1
1787    10.31124/advance.10048160.v2
1786    10.31124/advance.10048160.v3
3715    10.31124/advance.10048160.v4
653        10.31124/advance.10050131
1789    10.31124/advance.10050131.v1
345        10.31124/advance.10050497
1788    10.31124/advance.10050497.v1
347        10.31124/advance.10055363
1790    10.31124/advance.10055363.v1
344        10.31124/advance.10055399
1791    10.31124/advance.10055399.v1
348        10.31124/advance.10096058
1

In [11]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
2,crossref::10.31124/advance.7038500.v1,Advance,crossref,10.31124/advance.7038500.v1,10.31124/advance.7038500.v1,https://doi.org/10.31124/advance.7038500.v1,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500.v1"", ""URL"": ..."
4,crossref::10.31124/advance.7038503.v1,Advance,crossref,10.31124/advance.7038503.v1,10.31124/advance.7038503.v1,https://doi.org/10.31124/advance.7038503.v1,https://advance.sagepub.com/articles/A_TROG_ma...,https://advance.sagepub.com/articles/A_TROG_ma...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A TROG masked.docUsing T.R.O.G to improve psyc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>A discussion on language from a theore...,A discussion on language from a theoretical pe...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038503.v1"", ""URL"": ..."
5,crossref::10.31124/advance.7038506.v1,Advance,crossref,10.31124/advance.7038506.v1,10.31124/advance.7038506.v1,https://doi.org/10.31124/advance.7038506.v1,https://advance.sagepub.com/articles/Are_We_Th...,https://advance.sagepub.com/articles/Are_We_Th...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Are We There Yet? Understanding Cultural Issue...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.o

In [12]:
# [./] matches either a dot or a slash
# v\d+ matches 'v' followed by one or more digits
# $ ensures this pattern is at the very end of the string
pattern = r'[./]v\d+$'
# pattern = r'v\d+$'
mask = ~df['doi'].str.contains(pattern, regex=True, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.31124/advance.7037639,Advance,crossref,10.31124/advance.7037639,10.31124/advance.7037639,https://doi.org/10.31124/advance.7037639,https://advance.sagepub.com/articles/The_Priva...,https://advance.sagepub.com/articles/The_Priva...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Privatization of Security and the Emergenc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2022-03-30,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;This study interrogates the p...,This study interrogates the participation of p...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Chinwokwu, Eke; Igbo, Emmanuel",None,None,"[{""affiliation"": [], ""family"": ""Chinwokwu"", ""g...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7037639"", ""URL"": ""ht..."
1,crossref::10.31124/advance.7038500,Advance,crossref,10.31124/advance.7038500,10.31124/advance.7038500,https://doi.org/10.31124/advance.7038500,https://advance.sagepub.com/articles/Unmet_nee...,https://advance.sagepub.com/articles/Unmet_nee...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,Unmet needs of ageing transgender and non-bina...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>An view of literature on transgender a...,An view of literature on transgender and non-b...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Broadway-Horner, Matt",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8834-7...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.7038500"", ""URL"": ""ht..."
3,crossref::10.31124/advance.7038503,Advance,crossref,10.31124/advance.7038503,10.31124/advance.7038503,https://doi.org/10.31124/advance.7038503,https://advance.sagepub.com/articles/A_TROG_ma...,https://advance.sagepub.com/articles/A_TROG_ma...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,A TROG masked.docUsing T.R.O.G to improve psyc...,None,None,None,None,posted-content,preprint,preprint,None,True,2018-09-04,2018-09-04,2018-09-04,2025-02-21,None,2018-09-04,None,2018-09-04,None,2018,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<

In [13]:
pattern = "10.31124/advance.14132306"

mask = df['doi'].str.contains(pattern, regex=False, na=False)
result = df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
2473,crossref::10.31124/advance.14132306,Advance,crossref,10.31124/advance.14132306,10.31124/advance.14132306,https://doi.org/10.31124/advance.14132306,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Effect of Healthcare Education on Future D...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-09,2024-02-22,2024-07-17,None,2021-03-09,None,2021-03-09,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306"", ""URL"": ""h..."
2475,crossref::10.31124/advance.14132306.v1,Advance,crossref,10.31124/advance.14132306.v1,10.31124/advance.14132306.v1,https://doi.org/10.31124/advance.14132306.v1,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The effect of healthcare education on students...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-04,2021-03-04,2024-02-22,2024-02-23,None,2021-03-04,None,2021-03-04,None,2021,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>&lt;p&gt;In competitive education test...,In competitive education test scores and scien...,"[{""URL"": ""https://ndownloader.figshare.com/fil...",None,"Nowak, Ewa; Barciszewska, Anna-Maria; Lind, Ge...",None,None,"[{""affiliation"": [], ""family"": ""Nowak"", ""given...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,63,None,"{""DOI"": ""10.31124/advance.14132306.v1"", ""URL"":..."
2477,crossref::10.31124/advance.14132306.v2,Advance,crossref,10.31124/advance.14132306.v2,10.31124/advance.14132306.v2,https://doi.org/10.31124/advance.14132306.v2,https://advance.sagepub.com/doi/full/10.31124/...,https://advance.sagepub.com/doi/full/10.31124/...,10.31124,179,None,None,crossref,SAGE Publications,None,None,None,None,The Effect of Healthcare Education on Future D...,None,None,None,None,posted-content,preprint,preprint,None,True,2021-03-09,2021-03-09,2024-02-22,2024-02-23,None,2021-03-09,None,2021-03-09,None,2021,issued_date,posted_date,None,N

In [14]:
advance_parent_df, advance_parent_summary = analyze_parent_data("Advance", full_parent_df)


 CONSOLIDATED ANALYSIS: ADVANCE
  > Total Parent Groups:     2,336
  > Total Versions Found:    4,308
  > Unique Parent DOIs:      2,336
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Advance:
-----------------------------------
  1 version(s):     653 groups
  2 version(s):     1,477 groups
  3 version(s):     161 groups
  4 version(s):     33 groups
  5 version(s):     9 groups
  7 version(s):     1 groups
  8 version(s):     1 groups
  26 version(s):    1 groups

  Average versions per parent: 1.84

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Advance                  2271
SSRN                       28
Research Square             7
SocArXiv                    6
Preprints.org               5
Advance; ResearchGate       4
ResearchGate                3
AgEcon Search               2
arXiv                       2
Authorea Inc.               2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKE

# AfricArXiv

In [15]:
AfricArXiv_df, AfricArXiv_summary = get_server_data("AfricArXiv")


 SERVER ANALYSIS: AFRICARXIV
  > Files found:    2
  > Raw records:    2190
OK


In [16]:
AfricArXiv_parent_df, AfricArXiv_parent_summary = analyze_parent_data("AfricArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: AFRICARXIV
  > Total Parent Groups:     1,578
  > Total Versions Found:    2,107
  > Unique Parent DOIs:      1,578
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AfricArXiv:
-----------------------------------
  1 version(s):     1,142 groups
  2 version(s):     391 groups
  3 version(s):     28 groups
  4 version(s):     10 groups
  6 version(s):     1 groups
  8 version(s):     3 groups
  9 version(s):     2 groups
  11 version(s):    1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AfricArXiv                 1566
Open Science Framework        3
SSRN                          3
ResearchGate                  2
SocArXiv                      1
Advance                       1
Humanities Commons CORE       1
Zenodo                        1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
d

# AgEcon Search

In [17]:
AgEcon_Search_df, AgEcon_Search_summary = get_server_data("AgEcon_Search")


 SERVER ANALYSIS: AGECON_SEARCH
  > Files found:    2
  > Raw records:    188173
OK


In [18]:
# for x in AgEcon_Search_df.sample(1).iloc[0]:
#     print(x)

In [19]:
AgEcon_parent_df, AgEcon_parent_summary = analyze_parent_data("AgEcon Search", full_parent_df)


 CONSOLIDATED ANALYSIS: AGECON SEARCH
  > Total Parent Groups:     153,387
  > Total Versions Found:    167,422
  > Unique Parent DOIs:      153,387
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AgEcon Search:
-----------------------------------
  1 version(s):     145,430 groups
  2 version(s):     6,904 groups
  3 version(s):     598 groups
  4 version(s):     142 groups
  5 version(s):     63 groups
  6 version(s):     49 groups
  7 version(s):     30 groups
  8 version(s):     21 groups
  9 version(s):     16 groups
  10 version(s):    14 groups
  11 version(s):    7 groups
  12 version(s):    11 groups
  13 version(s):    4 groups
  14 version(s):    9 groups
  15 version(s):    10 groups
  16 version(s):    3 groups
  17 version(s):    3 groups
  18 version(s):    7 groups
  19 version(s):    3 groups
  20 version(s):    2 groups
  21 version(s):    2 groups
  22 version(s):    4 groups
  23 version(s):    3 groups
  24 version(s

In [20]:
AgEcon_Search_df[AgEcon_Search_df['type_backend_raw']=='Other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
166567,datacite::10.22004/ag.econ.333722,AgEcon Search,datacite,10.22004/ag.econ.333722,10.22004/ag.econ.333722,https://doi.org/10.22004/ag.econ.333722,https://ageconsearch.umn.edu/record/333722,https://ageconsearch.umn.edu/record/333722,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Farmers' risk exposition and its drivers,None,None,None,en,Other,Working or Discussion Paper,Other,None,None,2023-04-06,None,None,None,2025-09-30,None,2023-04-06,None,None,2019,published_year,None,None,None,None,None,"[{""description"": ""The analysis of income risk ...",The analysis of income risk is the basis for f...,None,None,"Duden, C.; Offermann, F.",None,None,"[{""affiliation"": [], ""familyName"": ""Duden"", ""g...",[],None,[],None,None,"[{""subject"": ""Farm Management""}, {""subject"": ""...",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id,0,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content..."
166568,datacite::10.22004/ag.econ.333733,AgEcon Search,datacite,10.22004/ag.econ.333733,10.22004/ag.econ.333733,https://doi.org/10.22004/ag.econ.333733,https://ageconsearch.umn.edu/record/333733,https://ageconsearch.umn.edu/record/333733,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Front matter,None,None,None,en,Other,Journal Article,Other,None,None,2023-04-06,None,None,None,2023-04-06,None,2023-04-06,None,None,2018,published_year,None,None,None,None,None,[],None,None,None,Australian Journal Of Agricultural And Resourc...,None,None,"[{""affiliation"": [], ""name"": ""Australian Journ...",[],None,[],None,None,"[{""subject"": ""Agricultural and Food Policy""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id,0,"{""client"": {""data"": {""id"": ""tind.agecon"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content..."
166569,datacite::10.22004/ag.econ.333734,AgEcon Search,datacite,10.22004/ag.econ.333734,10.22004/ag.econ.333734,https://doi.org/10.22004/ag.econ.333734,https://ageconsearch.umn.edu/record/333734,https://ageconsearch.umn.edu/record/333734,10.22004,None,tind.agecon,tind,datacite,Unknown,None,None,None,None,Effects of supermarket monopsony pricing on ag...,None,None,None,en,Other,Journal Article,Other,None,None,2023-04-06,None,None,None,2023-04-06,None,2023-04-06,None,None,2018,published_year,None,None,None,None,None,"[{""description"": ""Potential effects of alleged...",Potential effects of alleged monopsony pricing...,None,None,"Freebairn, John",None,None,"[{""affiliation"": [], ""familyName"": ""Freebairn""...",[],None,[],None,None,"[{""subject"": ""International Relations/Trade""},...",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id,0,"{""client"": {""data"": {""id"":

# AgriRxiv

In [21]:
AgriRxiv_df, AgriRxiv_summary = get_server_data("AgriRxiv")


 SERVER ANALYSIS: AGRIRXIV
  > Files found:    1
  > Raw records:    818
OK


In [22]:
AgriRxiv_parent_df, AgriRxiv_parent_summary = analyze_parent_data("AgriRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: AGRIRXIV
  > Total Parent Groups:     777
  > Total Versions Found:    909
  > Unique Parent DOIs:      777
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AgriRxiv:
-----------------------------------
  1 version(s):     672 groups
  2 version(s):     90 groups
  3 version(s):     8 groups
  4 version(s):     4 groups
  5 version(s):     1 groups
  6 version(s):     2 groups

  Average versions per parent: 1.17

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AgriRxiv                            713
Open Science Framework               18
AgriRxiv; IndiaRxiv                  10
IndiaRxiv                             5
AgriRxiv; Open Science Framework      5
AgEcon Search                         4
INA-Rxiv                              4
SSRN                                  3
Research Square                       3
PeerJ Preprints                       1

 TOP V

# AIJR Preprints

In [23]:
AIJR_Preprints_df, AIJR_Preprints_summary = get_server_data("AIJR_Preprints")


 SERVER ANALYSIS: AIJR_PREPRINTS
  > Files found:    1
  > Raw records:    143
OK


In [24]:
AIJR_parent_df, AIJR_parent_summary = analyze_parent_data("AIJR Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: AIJR PREPRINTS
  > Total Parent Groups:     141
  > Total Versions Found:    150
  > Unique Parent DOIs:      141
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AIJR Preprints:
-----------------------------------
  1 version(s):     134 groups
  2 version(s):     5 groups
  3 version(s):     2 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AIJR Preprints                  134
ResearchGate                      3
ScienceOpen Preprints             1
AIJR Preprints; ResearchGate      1
AfricArXiv                        1
arXiv                             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21467/preprints    141



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.aijr.org    141


 COMPLETED: AIJR PREPRINTS


# AMRC Open Research

In [25]:
AMRC_Open_Research_df, AMRC_Open_Research_summary = get_server_data("AMRC_Open_Research")


 SERVER ANALYSIS: AMRC_OPEN_RESEARCH
  > Files found:    2
  > Raw records:    102
OK


In [26]:
AMRC_Open_Research_df.sort_values(by='title')

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
98,crossref::10.12688/healthopenres.13924.1,AMRC Open Research,crossref,10.12688/healthopenres.13924.1,10.12688/healthopenres.13924.1,https://doi.org/10.12688/healthopenres.13924.1,https://healthopenresearch.org/articles/7-17/v1,https://healthopenresearch.org/articles/7-17/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Health Open Research,None,None,2753-6416,A Protocol for Systematic Review of Prognostic...,None,None,None,en,journal-article,None,journal-article,None,False,2025-11-21,None,2025-11-21,2025-11-21,None,2025-10-10,None,2025-10-10,2025-10-10,2025.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns3:p>Background Survivors of adolescent and ...,Background Survivors of adolescent and young a...,"[{""URL"": ""https://healthopenresearch.org/artic...",https://healthopenresearch.org/articles/7-17/v...,"Guolla, Louise; Mbuagbaw, Lawrence; Ma, Jinhui...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-2199-6...",None,None,"[{""award"": [""CVG-186971""], ""award-info"": [{""aw...",CIHR Vanier Canada Graduate Scholarship,1.0,None,None,None,0,None,None,0,58,"[{""DOI"": ""10.1161/CIRCULATIONAHA.119.041403"", ...",None,,,False,None,,None,None,,None,None,https://doi.org/10.12688/healthopenres.crossma...,issn,0,None,"{""DOI"": ""10.12688/healthopenres.13924.1"", ""ISS..."
17,crossref::10.12688/amrcopenres.12936.2,AMRC Open Research,crossref,10.12688/amrcopenres.12936.2,10.12688/amrcopenres.12936.2,https://doi.org/10.12688/amrcopenres.12936.2,https://amrcopenresearch.org/articles/2-29/v2,https://amrcopenresearch.org/articles/2-29/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,AMRC Open Research,None,None,2517-6900,A collaborative approach to exercise provision...,None,None,None,en,journal-article,None,journal-article,None,False,2021-04-01,None,2021-04-26,2026-02-27,None,2021-04-01,None,2021-04-01,2021-04-01,2021.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns3:p>\n <ns3:bold>Backgro...,Background: Exercise has been shown to be bene...,"[{""URL"": ""https://amrcopenresearch.org/article...",https://amrcopenresearch.org/articles/2-29/v2/pdf,"Jones, Julie; Alexander, Lyndsay; Hancock, Eli...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-1943-1...",None,None,"[{""award"": [""F-1901""], ""award-info"": [{""award-...","Parkinson's UK; Chief Scientist Office, Scotland",2.0,None,None,None,1,None,None,1,62,"[{""DOI"": ""10.1002/mds.25945"", ""article-title"":...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,New version,10.21956/amrcopenres.14059.r26662;10.21956/amr...,"[{""DOI"": ""10.12688/amrcopenres.12936.1"", ""labe...",None,https://doi.org/10.12688/amrcopenres.crossmark...,issn,0,None,"

In [27]:
AMRC_parent_df, AMRC_parent_summary = analyze_parent_data("AMRC Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: AMRC OPEN RESEARCH
  > Total Parent Groups:     69
  > Total Versions Found:    99
  > Unique Parent DOIs:      69
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for AMRC Open Research:
-----------------------------------
  1 version(s):     43 groups
  2 version(s):     22 groups
  3 version(s):     4 groups

  Average versions per parent: 1.43

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
AMRC Open Research    69

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/healthopenres    35
10.12688/amrcopenres      34



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
healthopenresearch.org    41
amrcopenresearch.org      28


 COMPLETED: AMRC OPEN RESEARCH



# APSA Preprints

In [28]:
APSA_Preprints_df, APSA_Preprints_summary = get_server_data("APSA_Preprints")


 SERVER ANALYSIS: APSA_PREPRINTS
  > Files found:    1
  > Raw records:    1470
OK


In [29]:
APSA_parent_df, APSA_parent_summary = analyze_parent_data("APSA Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: APSA PREPRINTS
  > Total Parent Groups:     1,130
  > Total Versions Found:    1,497
  > Unique Parent DOIs:      1,130
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for APSA Preprints:
-----------------------------------
  1 version(s):     908 groups
  2 version(s):     149 groups
  3 version(s):     45 groups
  4 version(s):     14 groups
  5 version(s):     4 groups
  6 version(s):     3 groups
  7 version(s):     3 groups
  8 version(s):     1 groups
  9 version(s):     1 groups
  10 version(s):    1 groups
  14 version(s):    1 groups

  Average versions per parent: 1.32

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
APSA Preprints                  1089
SSRN                              27
SocArXiv                           4
Open Science Framework             3
Qeios                              1
Cambridge Open Engage              1
APSA Preprints; Res

# Arabixiv

In [30]:
Arabixiv_df, Arabixiv_summary = get_server_data("Arabixiv")


 SERVER ANALYSIS: ARABIXIV
  > Files found:    1
  > Raw records:    502
OK


In [31]:
Arabixiv_parent_df, Arabixiv_parent_summary = analyze_parent_data("Arabixiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ARABIXIV
  > Total Parent Groups:     476
  > Total Versions Found:    585
  > Unique Parent DOIs:      476
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Arabixiv:
-----------------------------------
  1 version(s):     383 groups
  2 version(s):     78 groups
  3 version(s):     14 groups
  4 version(s):     1 groups

  Average versions per parent: 1.23

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Arabixiv                            457
Frenxiv                              11
Arabixiv; Open Science Framework      4
ResearchGate                          2
Arabixiv; Frenxiv; SocArXiv           1
Open Science Framework                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31221/osf    471
10.17605/osf      5



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domai

# ARPHA Preprints

In [32]:
ARPHA_df, ARPHA_summary = get_server_data("ARPHA_Preprints")


 SERVER ANALYSIS: ARPHA_PREPRINTS
  > Files found:    1
  > Raw records:    890
OK


In [33]:
ARPHA_parent_df, ARPHA_parent_summary = analyze_parent_data("ARPHA Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ARPHA PREPRINTS
  > Total Parent Groups:     855
  > Total Versions Found:    881
  > Unique Parent DOIs:      855
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ARPHA Preprints:
-----------------------------------
  1 version(s):     830 groups
  2 version(s):     24 groups
  3 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ARPHA Preprints    851
SSRN                 1
bioRxiv              1
Research Square      1
Zenodo               1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.3897/arphapreprints    855



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.arphahub.com    855


 COMPLETED: ARPHA PREPRINTS



# ART-Dok

In [34]:
ART_df, ART_summary = get_server_data("ART-Dok")


 SERVER ANALYSIS: ART-DOK
  > Files found:    2
  > Raw records:    9653
OK


In [35]:
ART_parent_df, ART_parent_summary = analyze_parent_data("ART-Dok", full_parent_df)


 CONSOLIDATED ANALYSIS: ART-DOK
  > Total Parent Groups:     9,529
  > Total Versions Found:    9,654
  > Unique Parent DOIs:      9,529
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ART-Dok:
-----------------------------------
  1 version(s):     9,426 groups
  2 version(s):     91 groups
  3 version(s):     10 groups
  4 version(s):     1 groups
  12 version(s):    1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ART-Dok                    9527
Humanities Commons CORE       2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.11588/artdok    9529



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    9527
ub.uni-heidelberg.de              2


 COMPLETED: ART-DOK



# arXiv

In [36]:
# arXiv_df, arXiv_summary = get_server_data("arXiv")

In [37]:
arXiv_parent_df, arXiv_parent_summary = analyze_parent_data("arXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ARXIV
  > Total Parent Groups:     2,857,685
  > Total Versions Found:    2,939,445
  > Unique Parent DOIs:      2,857,685
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for arXiv:
-----------------------------------
  1 version(s):     2,784,275 groups
  2 version(s):     66,944 groups
  3 version(s):     5,329 groups
  4 version(s):     772 groups
  5 version(s):     223 groups
  6 version(s):     74 groups
  7 version(s):     29 groups
  8 version(s):     13 groups
  9 version(s):     10 groups
  10 version(s):    2 groups
  11 version(s):    5 groups
  12 version(s):    4 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  19 version(s):    1 groups
  69 version(s):    1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
arXiv                                         2821925
SSRN                         

# Authorea Inc.

In [38]:
Authorea_df, Authorea_summary = get_server_data("Authorea_Inc.")


 SERVER ANALYSIS: AUTHOREA_INC.
  > Files found:    1
  > Raw records:    65450
OK


In [39]:
Authorea_parent_df, Authorea_parent_summary = analyze_parent_data("Authorea Inc.", full_parent_df)


 CONSOLIDATED ANALYSIS: AUTHOREA INC.
  > Total Parent Groups:     56,167
  > Total Versions Found:    60,446
  > Unique Parent DOIs:      56,167
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Authorea Inc.:
-----------------------------------
  1 version(s):     52,683 groups
  2 version(s):     2,976 groups
  3 version(s):     375 groups
  4 version(s):     90 groups
  5 version(s):     17 groups
  6 version(s):     10 groups
  7 version(s):     4 groups
  8 version(s):     2 groups
  9 version(s):     2 groups
  11 version(s):    1 groups
  12 version(s):    1 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  16 version(s):    1 groups
  17 version(s):    1 groups
  19 version(s):    1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Authorea Inc.                           55050
Research Square                          

# Beilstein Archives

In [40]:
Beilstein_df, Beilstein_summary = get_server_data("Beilstein_Archives")


 SERVER ANALYSIS: BEILSTEIN_ARCHIVES
  > Files found:    1
  > Raw records:    697
OK


In [41]:
Beilstein_parent_df, Beilstein_parent_summary = analyze_parent_data("Beilstein Archives", full_parent_df)


 CONSOLIDATED ANALYSIS: BEILSTEIN ARCHIVES
  > Total Parent Groups:     694
  > Total Versions Found:    702
  > Unique Parent DOIs:      694
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Beilstein Archives:
-----------------------------------
  1 version(s):     687 groups
  2 version(s):     6 groups
  3 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Beilstein Archives    687
arXiv                   5
SSRN                    1
ChemRxiv                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.3762/bxiv    694



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
beilstein-archives.org    694


 COMPLETED: BEILSTEIN ARCHIVES



# BioHackrXiv

In [42]:
BioHackrXiv_df, BioHackrXiv_summary = get_server_data("BioHackrXiv")


 SERVER ANALYSIS: BIOHACKRXIV
  > Files found:    1
  > Raw records:    139
OK


In [43]:
BioHackrXiv_parent_df, BioHackrXiv_parent_summary = analyze_parent_data("BioHackrXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BIOHACKRXIV
  > Total Parent Groups:     136
  > Total Versions Found:    140
  > Unique Parent DOIs:      136
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for BioHackrXiv:
-----------------------------------
  1 version(s):     132 groups
  2 version(s):     4 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
BioHackrXiv                  135
BioHackrXiv; ResearchGate      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.37044/osf    136



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    136


 COMPLETED: BIOHACKRXIV



In [44]:
BioHackrXiv_parent_df[BioHackrXiv_parent_df['total_versions']==2]['version_dois'].sample(3).iloc[0]

'10.13140/rg.2.2.22058.34246; 10.37044/osf.io/5j7cm'

# bioRxiv

In [45]:
bioRxiv_df, bioRxiv_summary = get_server_data("bioRxiv")


 SERVER ANALYSIS: BIORXIV
  > Files found:    1
  > Raw records:    306948
OK


In [46]:
bioRxiv_parent_df, bioRxiv_parent_summary = analyze_parent_data("bioRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BIORXIV
  > Total Parent Groups:     303,393
  > Total Versions Found:    331,893
  > Unique Parent DOIs:      303,393
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for bioRxiv:
-----------------------------------
  1 version(s):     285,068 groups
  2 version(s):     14,112 groups
  3 version(s):     1,318 groups
  4 version(s):     358 groups
  5 version(s):     2,055 groups
  6 version(s):     443 groups
  7 version(s):     34 groups
  8 version(s):     4 groups
  12 version(s):    1 groups

  Average versions per parent: 1.09

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
bioRxiv            285372
eLife                9497
Research Square      3755
SSRN                 2616
arXiv; bioRxiv        413
Authorea Inc.         395
arXiv                 393
F1000Research         163
HAL                    90
Preprints.org          85

 TOP VALUES FOR: DOI_PREFIX_

# BodoArXiv

In [47]:
BodoArXiv_df, BodoArXiv_summary = get_server_data("BodoArXiv")


 SERVER ANALYSIS: BODOARXIV
  > Files found:    1
  > Raw records:    165
OK


In [48]:
BodoArXiv_parent_df, BodoArXiv_parent_summary = analyze_parent_data("BodoArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: BODOARXIV
  > Total Parent Groups:     149
  > Total Versions Found:    161
  > Unique Parent DOIs:      149
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for BodoArXiv:
-----------------------------------
  1 version(s):     139 groups
  2 version(s):     9 groups
  4 version(s):     1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
BodoArXiv      147
Law Archive      1
Zenodo           1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.34055/osf    146
10.31219/osf      3



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    149


 COMPLETED: BODOARXIV



# Cambridge Open Engage

In [49]:
Cambridge_df, Cambridge_summary = get_server_data("Cambridge_Open_Engage")


 SERVER ANALYSIS: CAMBRIDGE_OPEN_ENGAGE
  > Files found:    1
  > Raw records:    3090
OK


In [50]:
Cambridge_parent_df, Cambridge_parent_summary = analyze_parent_data("Cambridge Open Engage", full_parent_df)


 CONSOLIDATED ANALYSIS: CAMBRIDGE OPEN ENGAGE
  > Total Parent Groups:     1,875
  > Total Versions Found:    2,976
  > Unique Parent DOIs:      1,875
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Cambridge Open Engage:
-----------------------------------
  1 version(s):     1,430 groups
  2 version(s):     266 groups
  3 version(s):     96 groups
  4 version(s):     29 groups
  5 version(s):     15 groups
  6 version(s):     6 groups
  7 version(s):     5 groups
  8 version(s):     4 groups
  9 version(s):     2 groups
  10 version(s):    1 groups
  11 version(s):    3 groups
  12 version(s):    5 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    2 groups
  22 version(s):    1 groups
  26 version(s):    1 groups
  27 version(s):    1 groups
  32 version(s):    1 groups
  65 version(s):    1 groups

  Average versions per parent: 1.59

 MOST RE

# CERN document server

In [51]:
CERN_df, CERN_summary = get_server_data("CERN_document_server")


 SERVER ANALYSIS: CERN_DOCUMENT_SERVER
  > Files found:    2
  > Raw records:    2631
OK


In [52]:
CERN_df, CERN_summary = get_server_data("CERN_document_server")


 SERVER ANALYSIS: CERN_DOCUMENT_SERVER
  > Files found:    2
  > Raw records:    2631
OK


In [53]:
CERN_parent_df, CERN_parent_summary = analyze_parent_data("CERN document server", full_parent_df)


 CONSOLIDATED ANALYSIS: CERN DOCUMENT SERVER
  > Total Parent Groups:     975
  > Total Versions Found:    1,670
  > Unique Parent DOIs:      697
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CERN document server:
-----------------------------------
  1 version(s):     433 groups
  2 version(s):     458 groups
  3 version(s):     52 groups
  4 version(s):     15 groups
  5 version(s):     7 groups
  6 version(s):     5 groups
  7 version(s):     3 groups
  8 version(s):     1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.71

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CERN document server                                   797
arXiv                                                  170
HAL                                                      2
Munich Personal RePEc Archive                            2
CERN document server; arXiv                            

# ChemRxiv

In [54]:
ChemRxiv_df, ChemRxiv_summary = get_server_data("ChemRxiv")


 SERVER ANALYSIS: CHEMRXIV
  > Files found:    1
  > Raw records:    46475
OK


In [55]:
ChemRxiv_parent_df, ChemRxiv_parent_summary = analyze_parent_data("ChemRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: CHEMRXIV
  > Total Parent Groups:     36,128
  > Total Versions Found:    46,790
  > Unique Parent DOIs:      36,128
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ChemRxiv:
-----------------------------------
  1 version(s):     28,095 groups
  2 version(s):     6,242 groups
  3 version(s):     1,341 groups
  4 version(s):     303 groups
  5 version(s):     83 groups
  6 version(s):     25 groups
  7 version(s):     14 groups
  8 version(s):     7 groups
  9 version(s):     5 groups
  10 version(s):    2 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  16 version(s):    1 groups
  23 version(s):    1 groups
  31 version(s):    1 groups
  36 version(s):    1 groups

  Average versions per parent: 1.30

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ChemRxiv                  35656
Res

# CogPrints

In [56]:
CogPrints_df, CogPrints_summary = get_server_data("CogPrints")


 SERVER ANALYSIS: COGPRINTS
  > Files found:    1
  > Raw records:    1537
OK


In [57]:
# CogPrints_df

In [58]:
CogPrints_parent_df, CogPrints_parent_summary = analyze_parent_data("CogPrints", full_parent_df)


 CONSOLIDATED ANALYSIS: COGPRINTS
  > Total Parent Groups:     1,487
  > Total Versions Found:    1,505
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CogPrints:
-----------------------------------
  1 version(s):     1,470 groups
  2 version(s):     16 groups
  3 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CogPrints          1472
arXiv                 9
SSRN                  2
PsyArXiv              1
viXra                 1
CogPrints; HAL        1
PhilSci-Archive       1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    1487



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None             1030
cogprints.org     457


 COMPLETED: COGPRINTS



In [59]:
pattern = r'10.48550/arxiv'
# pattern = r'v\d+$'
mask = CogPrints_parent_df['parent_doi'].str.contains(pattern, regex=True, na=False)
result = CogPrints_parent_df[mask]
result

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend


In [60]:
pattern = r'10.48550/arxiv.physics/0001050'
# pattern = r'v\d+$'
mask = full_parent_df['parent_doi'].str.contains(pattern, regex=True, na=False)
result = full_parent_df[mask]
result

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers
4244908,datacite::10.48550/arxiv.physics/0001050,datacite::10.48550/arxiv.physics/0001050,arXiv,10.48550/arxiv.physics/0001050,https://arxiv.org/abs/physics/0001050,Statistical mechanics of neocortical interacti...,"Ingber, Lester",2000-01-23,2000,datacite::10.48550/arxiv.physics/0001050,10.48550/arxiv.physics/0001050,arXiv (1),1,2000-01-23,2000-01-23,arXiv


# CoP

In [61]:
CoP_df, CoP_summary = get_server_data("CoP")


 SERVER ANALYSIS: COP
  > Files found:    1
  > Raw records:    30
OK


In [62]:
CoP_parent_df, CoP_parent_summary = analyze_parent_data("CoP", full_parent_df)


 CONSOLIDATED ANALYSIS: COP
  > Total Parent Groups:     29
  > Total Versions Found:    30
  > Unique Parent DOIs:      29
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CoP:
-----------------------------------
  1 version(s):     28 groups
  2 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CoP    29

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31219/osf    29



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    29


 COMPLETED: COP



# Covid-19 Preprints

In [63]:
Covid_df, Covid_summary = get_server_data("Covid-19_Preprints")


 SERVER ANALYSIS: COVID-19_PREPRINTS
  > Files found:    1
  > Raw records:    647
OK


In [64]:
Covid_parent_df, Covid_parent_summary = analyze_parent_data("Covid-19 Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: COVID-19 PREPRINTS
  > Total Parent Groups:     293
  > Total Versions Found:    647
  > Unique Parent DOIs:      293
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Covid-19 Preprints:
-----------------------------------
  1 version(s):     262 groups
  2 version(s):     14 groups
  3 version(s):     2 groups
  4 version(s):     5 groups
  5 version(s):     1 groups
  6 version(s):     1 groups
  8 version(s):     1 groups
  9 version(s):     1 groups
  16 version(s):    1 groups
  21 version(s):    1 groups
  24 version(s):    1 groups
  33 version(s):    1 groups
  73 version(s):    1 groups
  136 version(s):   1 groups

  Average versions per parent: 2.21

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Covid-19 Preprints    292
PREPRINTS.RU            1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.

# CrimRxiv

In [65]:
CrimRxiv_df, CrimRxiv_summary = get_server_data("CrimRxiv")


 SERVER ANALYSIS: CRIMRXIV
  > Files found:    1
  > Raw records:    2838
OK


In [66]:
CrimRxiv_parent_df, CrimRxiv_parent_summary = analyze_parent_data("CrimRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: CRIMRXIV
  > Total Parent Groups:     2,570
  > Total Versions Found:    2,713
  > Unique Parent DOIs:      2,570
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CrimRxiv:
-----------------------------------
  1 version(s):     2,443 groups
  2 version(s):     115 groups
  3 version(s):     10 groups
  4 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CrimRxiv                  2541
CrimRxiv; SocArXiv           8
CrimRxiv; arXiv              5
SSRN                         4
SocArXiv                     4
arXiv                        2
Open Science Framework       2
Research Square              2
CrimRxiv; ResearchGate       1
PsyArXiv                     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21428/cb     

In [67]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='component']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
1707,crossref::10.21428/cb6ab371.b8929691/7cc1f2de,CrimRxiv,crossref,10.21428/cb6ab371.b8929691/7cc1f2de,10.21428/cb6ab371.b8929691/7cc1f2de,https://doi.org/10.21428/cb6ab371.b8929691/7cc...,https://www.crimrxiv.com/pub/pag3wtd8,https://www.crimrxiv.com/pub/pag3wtd8,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,Open access to journal articles of the America...,None,None,None,None,component,None,component,None,False,2023-10-31,None,2023-10-31,2025-11-23,None,2023-10-31,None,2023-10-31,None,2023.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacques, Scott",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-2089-4...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.b8929691/7cc1f2de"",..."


In [68]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='book-chapter']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
2162,crossref::10.21428/cb6ab371.a4e2c432,CrimRxiv,crossref,10.21428/cb6ab371.a4e2c432,10.21428/cb6ab371.a4e2c432,https://doi.org/10.21428/cb6ab371.a4e2c432,https://www.crimrxiv.com/pub/g5531dv0,https://www.crimrxiv.com/pub/g5531dv0,10.21428,9621,None,None,crossref,PubPub,Criminology's Public Domain,None,None,None,Community Organization and Juvenile Delinquenc...,None,None,None,None,book-chapter,None,book-chapter,None,False,2024-11-03,None,2024-11-03,2024-11-03,None,2024-11-03,None,2024-11-03,2024-11-03,2024.0,issued_date,None,None,None,None,None,None,None,None,None,"Park, Robert E.",None,None,"[{""affiliation"": [], ""family"": ""Park"", ""given""...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.a4e2c432"", ""URL"": ""..."
2271,crossref::10.21428/cb6ab371.a03e79ee,CrimRxiv,crossref,10.21428/cb6ab371.a03e79ee,10.21428/cb6ab371.a03e79ee,https://doi.org/10.21428/cb6ab371.a03e79ee,https://www.crimrxiv.com/pub/1k60zvag,https://www.crimrxiv.com/pub/1k60zvag,10.21428,9621,None,None,crossref,PubPub,Criminology's Public Domain,None,None,None,An Essay [Excerpt] on Crimes and Punishments,None,None,None,None,book-chapter,None,book-chapter,None,False,2025-01-08,None,2025-01-08,2025-01-09,None,2025-01-08,None,2025-01-08,2025-01-08,2025.0,issued_date,None,None,None,None,None,None,None,None,None,"Beccaria, Cesare",None,None,"[{""affiliation"": [], ""family"": ""Beccaria"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-version-of"": [{""asserted-by"": ""subject"", ...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.a03e79ee"", ""URL"": ""..."


In [69]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='peer-review']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
1014,crossref::10.21428/cb6ab371.33c9e126,CrimRxiv,crossref,10.21428/cb6ab371.33c9e126,10.21428/cb6ab371.33c9e126,https://doi.org/10.21428/cb6ab371.33c9e126,https://www.crimrxiv.com/pub/s4okkv64,https://www.crimrxiv.com/pub/s4okkv64,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review 1 of ""Treating Criminal Justice-Involve...",None,None,None,None,peer-review,None,peer-review,None,False,2022-05-24,None,2022-05-24,2024-03-03,None,2022-05-24,None,2022-05-24,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Windle, James",None,None,"[{""affiliation"": [], ""family"": ""Windle"", ""give...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.33c9e126"", ""URL"": ""..."
1015,crossref::10.21428/cb6ab371.c4d02d0a,CrimRxiv,crossref,10.21428/cb6ab371.c4d02d0a,10.21428/cb6ab371.c4d02d0a,https://doi.org/10.21428/cb6ab371.c4d02d0a,https://www.crimrxiv.com/pub/frlqbegb,https://www.crimrxiv.com/pub/frlqbegb,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review 2 of ""Doing death work: A mixed method ...",None,None,None,None,peer-review,None,peer-review,None,False,2022-05-24,None,2022-05-24,2024-03-03,None,2022-05-24,None,2022-05-24,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Brondolo, Elizabeth",None,None,"[{""affiliation"": [], ""family"": ""Brondolo"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.c4d02d0a"", ""URL"": ""..."
1321,crossref::10.21428/cb6ab371.f514a7ba,CrimRxiv,crossref,10.21428/cb6ab371.f514a7ba,10.21428/cb6ab371.f514a7ba,https://doi.org/10.21428/cb6ab371.f514a7ba,https://www.crimrxiv.com/pub/19r39l3l,https://www.crimrxiv.com/pub/19r39l3l,10.21428,9621,None,None,crossref,PubPub,None,None,None,None,"Review of ""Ranking the openness of criminology...",None,None,None,None,peer-review,None,peer-review,None,False,2022-10-06,None,2022-10-06,2025-05-14,None,2022-10-06,None,2022-10-06,None,2022.0,issued_date,None,None,None,None,None,None,None,None,None,"Wheeler, Andrew",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-2255-1...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,"{""is-review-of"": [{""asserted-by"": ""subject"", ""...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.f514a7ba"", ""URL"": ""..."
1330,crossref::10.21428/cb6ab371.e2488a45,CrimRxiv,crossref,10.21428/cb6ab371.e2488a45,10.21428/cb6ab371.e2488a45,https://doi.org/10.21428/cb6ab371.e2488a45,https://www.crimrxiv.com/pub/iumal6

In [70]:
CrimRxiv_df[CrimRxiv_df['type_backend_raw']=='journal-article']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.21428/cb6ab371.30868967,CrimRxiv,crossref,10.21428/cb6ab371.30868967,10.21428/cb6ab371.30868967,https://doi.org/10.21428/cb6ab371.30868967,https://crimrxiv.pubpub.org/pub/tn1vo8l2,https://crimrxiv.pubpub.org/pub/tn1vo8l2,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,"Bentham, Not Epicurus: The Relevance of Pleasu...",None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2025-11-23,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacques, Scott",Georgia State University,None,"[{""affiliation"": [{""name"": ""Georgia State Univ...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.30868967"", ""URL"": ""..."
1,crossref::10.21428/cb6ab371.aab5ffbe,CrimRxiv,crossref,10.21428/cb6ab371.aab5ffbe,10.21428/cb6ab371.aab5ffbe,https://doi.org/10.21428/cb6ab371.aab5ffbe,https://crimrxiv.pubpub.org/pub/oxbqg1op,https://crimrxiv.pubpub.org/pub/oxbqg1op,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,Proterrence &amp; Rule Illegitimacy in an Age ...,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-07,None,2020-07-07,2022-04-05,None,2020-07-07,None,2020-07-07,2020-07-07,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Jacobs, Bruce; Jacques, Scott","University of Texas, Dallas; Georgia State Uni...",None,"[{""affiliation"": [{""name"": ""University of Texa...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.aab5ffbe"", ""URL"": ""..."
2,crossref::10.21428/cb6ab371.9e0bdd09,CrimRxiv,crossref,10.21428/cb6ab371.9e0bdd09,10.21428/cb6ab371.9e0bdd09,https://doi.org/10.21428/cb6ab371.9e0bdd09,https://crimrxiv.pubpub.org/pub/zk9k26ba,https://crimrxiv.pubpub.org/pub/zk9k26ba,10.21428,9621,None,None,crossref,PubPub,CrimRxiv,None,None,None,La cybercriminalité,None,None,None,en,journal-article,None,journal-article,None,False,2020-07-08,None,2020-07-08,2023-08-16,None,2020-07-08,None,2020-07-08,2020-07-08,2020.0,issued_date,None,None,None,None,None,None,None,None,None,"Décary-Hétu, David",None,None,"[{""affiliation"": [], ""family"": ""Décary-Hétu"", ...",None,None,None,None,None,None,None,None,1,None,None,1,0,None,None,,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,25,None,"{""DOI"": ""10.21428/cb6ab371.9e0bdd09"", ""URL"": ""..."
3,crossref::10.21428/cb6ab371.0bcce0c6,CrimRxiv,crossref,10.21428/cb6ab371.0bcce0c6,10.21428/cb6ab371.0bcce0c6,https://doi.org/10.21428/cb6ab371.0bcce0c6,https://crimrxiv.pubpub.org/pub/gf7zjbgf,https://crimrxiv.pubpub.org/pub/gf7zjbgf,10.21428,9621,None

# CrossAsia-Repository

In [71]:
CrossAsia_df, CrossAsia_summary = get_server_data("CrossAsia-Repository")


 SERVER ANALYSIS: CROSSASIA-REPOSITORY
  > Files found:    2
  > Raw records:    479
OK


In [72]:
CrossAsia_parent_df, CrossAsia_parent_summary = analyze_parent_data("CrossAsia-Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: CROSSASIA-REPOSITORY
  > Total Parent Groups:     459
  > Total Versions Found:    479
  > Unique Parent DOIs:      459
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for CrossAsia-Repository:
-----------------------------------
  1 version(s):     455 groups
  2 version(s):     2 groups
  4 version(s):     1 groups
  16 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
CrossAsia-Repository    459

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.48796/20    459



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
repository.crossasia.org    459


 COMPLETED: CROSSASIA-REPOSITORY



In [73]:
CrossAsia_df[CrossAsia_df['subtype_backend_raw']=='blog_entry']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
40,datacite::10.48796/20240116-000,CrossAsia-Repository,datacite,10.48796/20240116-000,10.48796/20240116-000,https://doi.org/10.48796/20240116-000,https://repository.crossasia.org/receive/cross...,https://repository.crossasia.org/receive/cross...,10.48796,None,shkl.lofhsj,shkl,datacite,Fachinformationsdienst (FID) Asien,None,None,None,None,Im Banne Chinas: Der Sinologe Wolfgang Franke ...,None,None,None,de,Text,blog_entry,Text,None,None,2024-01-18,None,None,None,2025-02-17,None,2024-01-18,None,None,2024,published_year,None,None,None,CrossAsia Open Access Repository-Lizenz,https://repository.crossasia.org/content/right...,"[{""description"": ""Der Sinologe und Historiker ...",Der Sinologe und Historiker Wolfgang Franke (*...,None,None,"Messingschlager, Stefan; Platzek, Antje",None,None,"[{""affiliation"": [], ""familyName"": ""Messingsch...","[{""affiliation"": [], ""contributorType"": ""Hosti...",None,[],None,None,"[{""subject"": ""Franke, Wolfgang""}, {""subject"": ...",None,None,0,0,None,None,0,None,"[{""relatedIdentifier"": ""https://repository.cro...",,,False,None,,None,None,,None,None,None,client_id,3,"{""client"": {""data"": {""id"": ""shkl.lofhsj"", ""typ...","{""citationCount"": 0, ""container"": {""identifier..."
42,datacite::10.48796/20240326-000,CrossAsia-Repository,datacite,10.48796/20240326-000,10.48796/20240326-000,https://doi.org/10.48796/20240326-000,https://repository.crossasia.org/receive/cross...,https://repository.crossasia.org/receive/cross...,10.48796,None,shkl.lofhsj,shkl,datacite,Fachinformationsdienst (FID) Asien,None,None,None,None,Grundlagen und Konzept zur Erwerbung und Lizen...,None,None,None,de,Text,blog_entry,Text,None,None,2024-03-26,None,None,None,2025-02-17,None,2024-03-26,None,None,2024,published_year,None,None,None,CrossAsia Open Access Repository-Lizenz,https://repository.crossasia.org/content/right...,"[{""description"": ""Die Staatsbibliothek zu Berl...",Die Staatsbibliothek zu Berlin (SBB) betreut s...,None,None,FID Asien,None,None,"[{""affiliation"": [], ""familyName"": ""FID Asien""...","[{""affiliation"": [], ""contributorType"": ""Hosti...",None,[],None,None,"[{""subject"": ""Fachinformationsdienst""}, {""subj...",None,None,0,0,None,None,0,None,"[{""relatedIdentifier"": ""https://repository.cro...",,,False,None,,None,None,,None,None,None,client_id,3,"{""client"": {""data"": {""id"": ""shkl.lofhsj"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content..."
247,datacite::10.48796/20240722-000,CrossAsia-Repository,datacite,10.48796/20240722-000,10.48796/20240722-000,https://doi.org/10.48796/20240722-000,https://repository.crossasia.org/receive/cross...,https://repository.crossasia.org/receive/cross...,10.48796,None,shkl.lofhsj,shkl,datacite,Fachinformationsdienst (FID) Asien,None,None,None,None,Launching a fieldwork-based project

# Digital Access to Scholarship at Harvard (DASH) (Harvard University)

In [74]:
DASH_df, DASH_summary = get_server_data("Digital_Access_to_Scholarship_at_Harvard_(DASH)_(H")


 SERVER ANALYSIS: DIGITAL_ACCESS_TO_SCHOLARSHIP_AT_HARVARD_(DASH)_(H
  > Files found:    1
  > Raw records:    9703
OK


In [75]:
DASH_parent_df, DASH_parent_summary = analyze_parent_data("Digital Access to Scholarship at Harvard (DASH) (Harvard University)", full_parent_df)


 CONSOLIDATED ANALYSIS: DIGITAL ACCESS TO SCHOLARSHIP AT HARVARD (DASH) (HARVARD UNIVERSITY)
  > Total Parent Groups:     9,383
  > Total Versions Found:    9,554
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Digital Access to Scholarship at Harvard (DASH) (Harvard University):
-----------------------------------
  1 version(s):     9,224 groups
  2 version(s):     150 groups
  3 version(s):     6 groups
  4 version(s):     3 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Digital Access to Scholarship at Harvard (DASH) (Harvard University)                                         9235
SSRN                                                                                                          117
arXiv                                                                                                          1

# DSpace@MIT

In [76]:
DSpace_df, DSpace_summary = get_server_data("DSpace@MIT")


 SERVER ANALYSIS: DSPACE@MIT
  > Files found:    1
  > Raw records:    12661
OK


In [77]:
DSpace_parent_df, DSpace_parent_summary = analyze_parent_data("DSpace@MIT", full_parent_df)


 CONSOLIDATED ANALYSIS: DSPACE@MIT
  > Total Parent Groups:     10,420
  > Total Versions Found:    10,686
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for DSpace@MIT:
-----------------------------------
  1 version(s):     10,165 groups
  2 version(s):     244 groups
  3 version(s):     11 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
DSpace@MIT                                                              10244
arXiv                                                                     131
SSRN                                                                       26
RePEc: Research Papers in Economics                                         9
eLife                                                                       2
bioRxiv                                                                     2
DSpace@MIT; R

# E-LIS Repository

In [78]:
E_LIS_df, E_LIS_summary = get_server_data("E-LIS_Repository")


 SERVER ANALYSIS: E-LIS_REPOSITORY
  > Files found:    1
  > Raw records:    9128
OK


In [79]:
E_LIS_parent_df, E_LIS_parent_summary = analyze_parent_data("E-LIS Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: E-LIS REPOSITORY
  > Total Parent Groups:     8,794
  > Total Versions Found:    8,884
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for E-LIS Repository:
-----------------------------------
  1 version(s):     8,718 groups
  2 version(s):     66 groups
  3 version(s):     6 groups
  4 version(s):     4 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
E-LIS Repository                                           8773
SSRN                                                          6
Humanities Commons CORE                                       4
arXiv                                                         2
E-LIS Repository; Social Science Open Access Repository       2
Open Science Framework                                        1
engrXiv                                                      

# Earth and Space Science Open Archive

In [80]:
Earth_df, Earth_summary = get_server_data("Earth_and_Space_Science_Open_Archive")


 SERVER ANALYSIS: EARTH_AND_SPACE_SCIENCE_OPEN_ARCHIVE
  > Files found:    1
  > Raw records:    22748
OK


In [81]:
Earth_parent_df, Earth_parent_summary = analyze_parent_data("Earth and Space Science Open Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: EARTH AND SPACE SCIENCE OPEN ARCHIVE
  > Total Parent Groups:     19,185
  > Total Versions Found:    22,706
  > Unique Parent DOIs:      19,185
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Earth and Space Science Open Archive:
-----------------------------------
  1 version(s):     16,385 groups
  2 version(s):     2,251 groups
  3 version(s):     431 groups
  4 version(s):     88 groups
  5 version(s):     19 groups
  6 version(s):     7 groups
  7 version(s):     1 groups
  9 version(s):     1 groups
  10 version(s):    1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.18

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Earth and Space Science Open Archive           18854
arXiv                                            105
EGUsphere                                         58
Research Square                                   47
SSRN

# EarthArXiv

In [82]:
EarthArXiv_df, EarthArXiv_summary = get_server_data("EarthArXiv")


 SERVER ANALYSIS: EARTHARXIV
  > Files found:    2
  > Raw records:    6537
OK


In [83]:
EarthArXiv_parent_df, EarthArXiv_parent_summary = analyze_parent_data("EarthArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: EARTHARXIV
  > Total Parent Groups:     6,256
  > Total Versions Found:    6,821
  > Unique Parent DOIs:      6,256
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EarthArXiv:
-----------------------------------
  1 version(s):     5,718 groups
  2 version(s):     516 groups
  3 version(s):     18 groups
  4 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.09

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EarthArXiv                              6015
EarthArXiv; Open Science Framework       125
Research Square                           23
SSRN                                      23
Earth and Space Science Open Archive      13
EarthArXiv; arXiv                         12
EGUsphere                                 11
arXiv                                     10
EarthArXiv; ResearchGate                   7
ResearchGate     

# EasyChair preprint

In [84]:
EasyChair_df, EasyChair_summary = get_server_data("EasyChair_preprint")


 SERVER ANALYSIS: EASYCHAIR_PREPRINT
  > Files found:    1
  > Raw records:    620
OK


In [85]:
EasyChair_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.29007/hsh2,EasyChair preprint,crossref,10.29007/hsh2,10.29007/hsh2,https://doi.org/10.29007/hsh2,https://easychair.org/publications/preprint/1,https://easychair.org/publications/preprint/1,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Unification with Abstraction and Theory Instan...,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-05,None,2017-09-13,None,2017-09-13,None,2017,issued_date,None,None,None,None,None,<jats:p>This paper explores two new inference ...,This paper explores two new inference rules fo...,None,None,"Reger, Giles; Suda, Martin; Voronkov, Andrei",None,None,"[{""affiliation"": [], ""family"": ""Reger"", ""given...",None,None,None,None,None,None,None,None,2,None,None,2,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/hsh2"", ""ISSN"": [""2516-2314""]..."
1,crossref::10.29007/g4bq,EasyChair preprint,crossref,10.29007/g4bq,10.29007/g4bq,https://doi.org/10.29007/g4bq,https://easychair.org/publications/preprint/WjKW,https://easychair.org/publications/preprint/WjKW,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,"Reconstructing Turing's ""paper machine""",None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2022-04-04,None,2017-09-14,None,2017-09-14,None,2017,issued_date,None,None,None,None,None,<jats:p>It is an amazing fact that the very fi...,It is an amazing fact that the very first ches...,None,None,"Kasparov, Garry; Friedel, Frederic",None,None,"[{""affiliation"": [], ""family"": ""Kasparov"", ""gi...",None,None,None,None,None,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/g4bq"", ""ISSN"": [""2516-2314""]..."
2,crossref::10.29007/pjn4,EasyChair preprint,crossref,10.29007/pjn4,10.29007/pjn4,https://doi.org/10.29007/pjn4,https://easychair.org/publications/preprint/N2sl,https://easychair.org/publications/preprint/N2sl,10.29007,11545,None,None,crossref,EasyChair,EasyChair Preprints,None,None,2516-2314,Computation of Some Integer Sequences in Maple,None,None,None,None,report-series,None,report-series,None,False,2018-01-12,None,2018-01-22,2024-07-11,None,2017-11-20,None,2017-11-20,None,2017,issued_date,None,None,None,None,None,<jats:p>We consider some integer sequences con...,We consider some integer sequences connected w...,None,None,"Fan, W.L.; Jeffrey, David J.; Postma, Erik",None,None,"[{""affiliation"": [], ""family"": ""Fan"", ""given"":...",None,None,None,None,None,None,None,None,7,None,None,7,0,None,None,,,False,None,,None,None,,None,NaN,None,issn,14,None,"{""DOI"": ""10.29007/pjn4"", ""ISSN"": [""2516-2314""]..."
3,crossref::10.29007/g5sh,EasyChair preprint,crossref,10.29007/g5sh,1

In [86]:
EasyChair_parent_df, EasyChair_parent_summary = analyze_parent_data("EasyChair preprint", full_parent_df)


 CONSOLIDATED ANALYSIS: EASYCHAIR PREPRINT
  > Total Parent Groups:     571
  > Total Versions Found:    606
  > Unique Parent DOIs:      571
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EasyChair preprint:
-----------------------------------
  1 version(s):     539 groups
  2 version(s):     29 groups
  3 version(s):     3 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EasyChair preprint                  541
arXiv                                20
PeerJ Preprints                       3
ResearchGate                          2
EasyChair preprint; arXiv             2
EasyChair preprint; ResearchGate      2
HAL                                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.29007/g    10
10.29007/h     8
10.29007/k     8
10.29007/p     7
10.29007/z     7
10.29007/

# EcoEvoRxiv

In [87]:
EcoEvoRxiv_df, EcoEvoRxiv_summary = get_server_data("EcoEvoRxiv")


 SERVER ANALYSIS: ECOEVORXIV
  > Files found:    2
  > Raw records:    2886
OK


In [88]:
EcoEvoRxiv_parent_df, EcoEvoRxiv_parent_summary = analyze_parent_data("EcoEvoRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ECOEVORXIV
  > Total Parent Groups:     2,795
  > Total Versions Found:    2,896
  > Unique Parent DOIs:      2,795
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EcoEvoRxiv:
-----------------------------------
  1 version(s):     2,721 groups
  2 version(s):     62 groups
  3 version(s):     6 groups
  4 version(s):     2 groups
  5 version(s):     3 groups
  10 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EcoEvoRxiv                                         2752
Authorea Inc.                                        15
Research Square                                       7
SSRN                                                  5
eLife                                                 4
Open Science Framework                                2
EcoEvoRxiv; Zenodo                                    2
HAL

# EconStor Preprints

In [89]:
EconStor_df, EconStor_summary = get_server_data("EconStor_Preprints")

/tmp/ipykernel_3173/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: ECONSTOR_PREPRINTS
  > Files found:    3
  > Raw records:    71761
OK


In [90]:
EconStor_parent_df, EconStor_parent_summary = analyze_parent_data("EconStor Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ECONSTOR PREPRINTS
  > Total Parent Groups:     62,784
  > Total Versions Found:    69,018
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EconStor Preprints:
-----------------------------------
  1 version(s):     57,156 groups
  2 version(s):     5,127 groups
  3 version(s):     422 groups
  4 version(s):     63 groups
  5 version(s):     14 groups
  8 version(s):     1 groups
  12 version(s):    1 groups

  Average versions per parent: 1.10

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EconStor Preprints                                           57507
SSRN                                                          4269
RePEc: Research Papers in Economics                            544
AgEcon Search                                                  194
EconStor Preprints; RePEc: Research Papers in Economics        166
ResearchG

# ECSarXiv

In [91]:
ECSarXiv_df, ECSarXiv_summary = get_server_data("ECSarXiv")


 SERVER ANALYSIS: ECSARXIV
  > Files found:    1
  > Raw records:    314
OK


In [92]:
ECSarXiv_parent_df, ECSarXiv_parent_summary = analyze_parent_data("ECSarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ECSARXIV
  > Total Parent Groups:     299
  > Total Versions Found:    334
  > Unique Parent DOIs:      299
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ECSarXiv:
-----------------------------------
  1 version(s):     265 groups
  2 version(s):     33 groups
  3 version(s):     1 groups

  Average versions per parent: 1.12

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ECSarXiv                                     289
ECSarXiv; Open Science Framework               4
Preprints.org                                  1
SSRN                                           1
engrXiv                                        1
ECSarXiv; arXiv                                1
ECSarXiv; Open Science Framework; engrXiv      1
arXiv                                          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10

# EdArXiv

In [93]:
EdArXiv_df, EdArXiv_summary = get_server_data("EdArXiv")


 SERVER ANALYSIS: EDARXIV
  > Files found:    1
  > Raw records:    2547
OK


In [94]:
EdArXiv_parent_df, EdArXiv_parent_summary = analyze_parent_data("EdArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: EDARXIV
  > Total Parent Groups:     2,153
  > Total Versions Found:    2,474
  > Unique Parent DOIs:      2,153
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EdArXiv:
-----------------------------------
  1 version(s):     1,911 groups
  2 version(s):     192 groups
  3 version(s):     38 groups
  4 version(s):     2 groups
  5 version(s):     7 groups
  6 version(s):     1 groups
  7 version(s):     1 groups
  9 version(s):     1 groups

  Average versions per parent: 1.15

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EdArXiv                                         2077
Thesis Commons                                    11
EdArXiv; ResearchGate                              9
SSRN                                               9
arXiv                                              8
Open Science Framework                             8
EdArXiv; arXiv         

# EGUsphere

In [95]:
EGUsphere_df, EGUsphere_summary = get_server_data("EGUsphere")


 SERVER ANALYSIS: EGUSPHERE
  > Files found:    1
  > Raw records:    15253
OK


In [96]:
pattern = r'10.5194/amt-|10.5194/hess-'
# pattern = r'v\d+$'
mask = EGUsphere_df['doi'].str.contains(pattern, regex=True, na=False)
result = EGUsphere_df[mask]
result

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
263,crossref::10.5194/amt-2022-295,EGUsphere,crossref,10.5194/amt-2022-295,10.5194/amt-2022-295,https://doi.org/10.5194/amt-2022-295,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,Temperature dependent sensitivity of iodide ch...,None,None,None,None,posted-content,preprint,preprint,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<jats:p>Abstract. Iodide chemical ionization m...,Abstract. Iodide chemical ionization mass spec...,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,"[{""DOI"": ""10.13039/100004800"", ""award"": [""20RD...",California Air Resources Board; NOAA Center fo...,2.0,None,None,None,1,None,None,1,0,None,"{""has-comment"": [{""asserted-by"": ""subject"", ""i...",,10.5194/amt-15-4295-2022,True,None,,None,None,10.5194/amt-2022-295-rc1;10.5194/amt-2022-295-rc2,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295"", ""URL"": ""https:..."
264,crossref::10.5194/amt-2022-295-supplement,EGUsphere,crossref,10.5194/amt-2022-295-supplement,10.5194/amt-2022-295-supplement,https://doi.org/10.5194/amt-2022-295-supplement,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,Gases/In Situ Measurement/Instruments and Plat...,None,"Supplementary material to ""Temperature depende...",None,None,None,None,posted-content,other,other,None,True,2022-05-11,2022-05-11,2023-03-21,2026-02-28,None,2022-05-11,None,2022-05-11,None,2022,issued_date,posted_date,None,None,None,None,None,None,None,None,"Robinson, Michael A.; Neuman, J. Andrew; Huey,...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-0977-9...",None,None,None,None,NaN,None,None,None,1,None,None,1,0,None,"{""is-supplement-to"": [{""asserted-by"": ""subject...",,,False,None,,None,None,,None,NaN,None,prefix/primary_domain,2,None,"{""DOI"": ""10.5194/amt-2022-295-supplement"", ""UR..."
9390,crossref::10.5194/amt-2024-3967,EGUsphere,crossref,10.5194/amt-2024-3967,10.5194/amt-2024-3967,https://doi.org/10.5194/amt-2024-3967,https://egusphere.copernicus.org/preprints/202...,https://egusphere.copernicus.org/preprints/202...,10.5194,3145,None,None,crossref,Copernicus GmbH,None,None,"Others (Wind, Precipitation, Temperature, etc....",None,High-resolution maps of Arctic surface skin te...,None,None,None,None,posted-content,preprint,preprint,None,True,2025-03-19,2025-03-19,2025-09-24,2026-02-2

https://doi.org/10.5194/amt-2024-3967 it seems like when publish, the doi change to have the same patterns as the journal doi

In [97]:
EGUsphere_parent_df, EGUsphere_parent_summary = analyze_parent_data("EGUsphere", full_parent_df)


 CONSOLIDATED ANALYSIS: EGUSPHERE
  > Total Parent Groups:     10,181
  > Total Versions Found:    10,300
  > Unique Parent DOIs:      10,181
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EGUsphere:
-----------------------------------
  1 version(s):     10,069 groups
  2 version(s):     106 groups
  3 version(s):     5 groups
  4 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EGUsphere                               10144
arXiv                                      20
Research Square                             6
Earth and Space Science Open Archive        3
EarthArXiv                                  3
ResearchGate                                1
SSRN                                        1
Preprints.org                               1
EGUsphere; arXiv                            1
Zenodo                              

# Electron Colloquium Comput Complex

In [98]:
Electron_df, Electron_summary = get_server_data("Electron_Colloquium_Comput_Complex")


 SERVER ANALYSIS: ELECTRON_COLLOQUIUM_COMPUT_COMPLEX
  > Files found:    1
  > Raw records:    227
OK


In [99]:
Electron_parent_df, Electron_parent_summary = analyze_parent_data("Electron Colloquium Comput Complex", full_parent_df)


 CONSOLIDATED ANALYSIS: ELECTRON COLLOQUIUM COMPUT COMPLEX
  > Total Parent Groups:     139
  > Total Versions Found:    170
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Electron Colloquium Comput Complex:
-----------------------------------
  1 version(s):     109 groups
  2 version(s):     29 groups
  3 version(s):     1 groups

  Average versions per parent: 1.22

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Electron Colloquium Comput Complex    110
arXiv                                  29

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    139



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
dblp.uni-trier.de      134
eccc.weizmann.ac.il      5


 COMPLETED: ELECTRON COLLOQUIUM COMPUT COMPLEX



# eLife

In [100]:
elife_df, elife_summary = get_server_data("eLife")


 SERVER ANALYSIS: ELIFE
  > Files found:    2
  > Raw records:    29901
OK


In [101]:
elife_df[elife_df['type_backend_raw']=='journal-article']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.7554/elife.02516,eLife,crossref,10.7554/elife.02516,10.7554/elife.02516,https://doi.org/10.7554/elife.02516,http://elifesciences.org/lookup/doi/10.7554/eL...,http://elifesciences.org/lookup/doi/10.7554/eL...,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Correction: A diversity of localized timescale...,None,None,None,en,journal-article,None,journal-article,None,False,2014-08-22,None,2014-08-22,2026-03-17,None,2014-02-18,None,2014-02-18,2014-02-18,2014.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/3.0/,http://creativecommons.org/licenses/by/3.0/,None,None,None,None,"Chaudhuri, Rishidev; Bernacchia, Alberto; Wang...",None,None,"[{""affiliation"": [], ""family"": ""Chaudhuri"", ""g...",None,None,None,None,NaN,None,None,None,18,None,None,18,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.02516"", ""ISSN"": [""2050-..."
1,crossref::10.7554/elife.08172,eLife,crossref,10.7554/elife.08172,10.7554/elife.08172,https://doi.org/10.7554/elife.08172,http://elifesciences.org/content/4/e08127,http://elifesciences.org/content/4/e08127,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,The number of olfactory stimuli that humans ca...,None,None,None,None,journal-article,None,journal-article,None,False,2015-07-07,None,2015-07-07,2022-04-05,None,2015-07-07,None,2015-07-07,2015-07-07,2015.0,issued_date,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,None,None,issn,0,None,"{""DOI"": ""10.7554/elife.08172"", ""ISSN"": [""2050-..."
3,crossref::10.7554/elife.43558,eLife,crossref,10.7554/elife.43558,10.7554/elife.43558,https://doi.org/10.7554/elife.43558,https://elifesciences.org/articles/43558,https://elifesciences.org/articles/43558,10.7554,4374,None,None,crossref,"eLife Sciences Publications, Ltd",eLife,None,None,2050-084X,Rapid task-dependent tuning of the mouse olfac...,None,None,None,en,journal-article,None,journal-article,None,False,2019-02-06,None,2019-02-06,2026-04-14,None,2019-02-06,None,2019-02-06,2019-02-06,2019.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<jats:p>Adapting neural representation to rapi...,Adapting neural representation to rapidly chan...,"[{""URL"": ""https://cdn.elifesciences.org/articl...",https://cdn.elifesciences.org/articles/43558/e...,"Koldaeva, Anzhelika; Schaefer, Andreas T; Fuku...","Sensory and Behavioural Neuroscience Unit, Oki...",None,"[{""affiliation"": [{""name"": ""Sensory and Behavi...",None,None,"[{""DOI"": ""10.13039/501100004199"", ""doi-asserte...",Okinawa Institute of Science and Technology Gr...,5.0,None,Non

In [102]:
eLife_parent_df, eLife_parent_summary = analyze_parent_data("eLife", full_parent_df)


 CONSOLIDATED ANALYSIS: ELIFE
  > Total Parent Groups:     9,439
  > Total Versions Found:    9,710
  > Unique Parent DOIs:      9,439
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for eLife:
-----------------------------------
  1 version(s):     9,183 groups
  2 version(s):     246 groups
  3 version(s):     7 groups
  4 version(s):     1 groups
  5 version(s):     2 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
eLife                                                                   9424
arXiv                                                                      7
medRxiv                                                                    1
Digital Access to Scholarship at Harvard (DASH) (Harvard University)       1
HAL; eLife                                                                 1
arXiv; eLife                                    

# ELPUB (Universitat Wuppertal)

In [103]:
ELPUB_df, ELPUB_summary = get_server_data("ELPUB_(Universitat_Wuppertal)")


 SERVER ANALYSIS: ELPUB_(UNIVERSITAT_WUPPERTAL)
  > Files found:    2
  > Raw records:    41
OK


/tmp/ipykernel_3173/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [104]:
ELPUB_parent_df, ELPUB_parent_summary = analyze_parent_data("ELPUB (Universitat Wuppertal)", full_parent_df)


 CONSOLIDATED ANALYSIS: ELPUB (UNIVERSITAT WUPPERTAL)
  > Total Parent Groups:     41
  > Total Versions Found:    41
  > Unique Parent DOIs:      41
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ELPUB (Universitat Wuppertal):
-----------------------------------
  1 version(s):     41 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ELPUB (Universitat Wuppertal)    41

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.25926/x          2
10.25926/0qp3       1
10.25926/nrcz-jr    1
10.25926/nf         1
10.25926/z          1
10.25926/pdf        1
10.25926/9k6n       1
10.25926/cfrc-ad    1
10.25926/hk         1
10.25926/ysd        1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
elpub.bib.uni-wuppertal.de    41


 COMPLETED: ELPUB (UNIVERSITAT WUPPERT

# EmeRI

In [105]:
EmeRI_df, EmeRI_summary = get_server_data("EmeRI")


 SERVER ANALYSIS: EMERI
  > Files found:    1
  > Raw records:    8
OK


In [106]:
EmeRI_parent_df, EmeRI_parent_summary = analyze_parent_data("EmeRI", full_parent_df)


 CONSOLIDATED ANALYSIS: EMERI
  > Total Parent Groups:     8
  > Total Versions Found:    8
  > Unique Parent DOIs:      8
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EmeRI:
-----------------------------------
  1 version(s):     8 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EmeRI    8

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.21452/15    6
10.21452/23    2



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.ibict.br    8


 COMPLETED: EMERI



# Encyclopedia

In [107]:
Encyclopedia_df, Encyclopedia_summary = get_server_data("Encyclopedia")


 SERVER ANALYSIS: ENCYCLOPEDIA
  > Files found:    1
  > Raw records:    166
OK


In [108]:
Encyclopedia_parent_df, Encyclopedia_parent_summary = analyze_parent_data("Encyclopedia", full_parent_df)


 CONSOLIDATED ANALYSIS: ENCYCLOPEDIA
  > Total Parent Groups:     161
  > Total Versions Found:    167
  > Unique Parent DOIs:      161
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Encyclopedia:
-----------------------------------
  1 version(s):     155 groups
  2 version(s):     6 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Encyclopedia     160
Preprints.org      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.32545/encyclopedia    161



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
encyclopedia.pub    161


 COMPLETED: ENCYCLOPEDIA



# EnerarXiv

In [109]:
EnerarXiv_df, EnerarXiv_summary = get_server_data("EnerarXiv")


 SERVER ANALYSIS: ENERARXIV
  > Files found:    1
  > Raw records:    204
OK


In [110]:
EnerarXiv_parent_df, EnerarXiv_parent_summary = analyze_parent_data("EnerarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ENERARXIV
  > Total Parent Groups:     191
  > Total Versions Found:    196
  > Unique Parent DOIs:      191
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for EnerarXiv:
-----------------------------------
  1 version(s):     187 groups
  2 version(s):     3 groups
  3 version(s):     1 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
EnerarXiv                  188
arXiv                        1
SSRN                         1
EnerarXiv; ResearchGate      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.46855/20    191



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
enerarxiv.org    191


 COMPLETED: ENERARXIV



# engrXiv

In [111]:
engrXiv_df, engrXiv_summary = get_server_data("engrXiv")


 SERVER ANALYSIS: ENGRXIV
  > Files found:    2
  > Raw records:    4929
OK


In [112]:
engrXiv_parent_df, engrXiv_parent_summary = analyze_parent_data("engrXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: ENGRXIV
  > Total Parent Groups:     4,519
  > Total Versions Found:    5,028
  > Unique Parent DOIs:      4,519
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for engrXiv:
-----------------------------------
  1 version(s):     4,077 groups
  2 version(s):     394 groups
  3 version(s):     37 groups
  4 version(s):     6 groups
  5 version(s):     3 groups
  6 version(s):     1 groups
  7 version(s):     1 groups

  Average versions per parent: 1.11

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
engrXiv                            4238
Open Science Framework               66
SSRN                                 48
Open Science Framework; engrXiv      41
arXiv                                26
TechRxiv                             24
arXiv; engrXiv                       16
Preprints.org                        12
Research Square                      11
ResearchGa

# F1000Research

In [113]:
F1000Research_df, F1000Research_summary = get_server_data("F1000Research")


 SERVER ANALYSIS: F1000RESEARCH
  > Files found:    1
  > Raw records:    16873
OK


In [114]:
F1000Research_parent_df, F1000Research_parent_summary = analyze_parent_data("F1000Research", full_parent_df)


 CONSOLIDATED ANALYSIS: F1000RESEARCH
  > Total Parent Groups:     10,640
  > Total Versions Found:    16,273
  > Unique Parent DOIs:      10,640
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for F1000Research:
-----------------------------------
  1 version(s):     6,408 groups
  2 version(s):     3,177 groups
  3 version(s):     812 groups
  4 version(s):     176 groups
  5 version(s):     48 groups
  6 version(s):     10 groups
  7 version(s):     5 groups
  8 version(s):     2 groups
  9 version(s):     1 groups
  11 version(s):    1 groups

  Average versions per parent: 1.53

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
F1000Research           10622
F1000Research; arXiv        4
ResearchGate                3
SSRN                        2
arXiv                       2
Research Square             2
searchRxiv                  1
Authorea Inc.               1
F1000Research; SSRN  

# FocUS Archive

In [115]:
FocUS_df, FocUS_summary = get_server_data("FocUS_Archive")


 SERVER ANALYSIS: FOCUS_ARCHIVE
  > Files found:    1
  > Raw records:    83
OK


In [116]:
FocUS_parent_df, FocUS_parent_summary = analyze_parent_data("FocUS Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: FOCUS ARCHIVE
  > Total Parent Groups:     79
  > Total Versions Found:    119
  > Unique Parent DOIs:      79
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for FocUS Archive:
-----------------------------------
  1 version(s):     43 groups
  2 version(s):     34 groups
  3 version(s):     1 groups
  5 version(s):     1 groups

  Average versions per parent: 1.51

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
FocUS Archive                            43
FocUS Archive; Open Science Framework    36

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31225/osf    79



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    79


 COMPLETED: FOCUS ARCHIVE



# Frenxiv

In [117]:
Frenxiv_df, Frenxiv_summary = get_server_data("Frenxiv")


 SERVER ANALYSIS: FRENXIV
  > Files found:    1
  > Raw records:    179
OK


In [118]:
Frenxiv_parent_df, Frenxiv_parent_summary = analyze_parent_data("Frenxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: FRENXIV
  > Total Parent Groups:     102
  > Total Versions Found:    116
  > Unique Parent DOIs:      102
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Frenxiv:
-----------------------------------
  1 version(s):     88 groups
  2 version(s):     14 groups

  Average versions per parent: 1.14

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Frenxiv                   94
SSRN                       4
Open Science Framework     2
Frenxiv; ResearchGate      1
EdArXiv                    1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31226/osf    102



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    102


 COMPLETED: FRENXIV



# Gates Open Research

In [119]:
Gates_df, Gates_summary = get_server_data("Gates_Open_Research")


 SERVER ANALYSIS: GATES_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    863
OK


In [120]:
Gates_parent_df, Gates_parent_summary = analyze_parent_data("Gates Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: GATES OPEN RESEARCH
  > Total Parent Groups:     502
  > Total Versions Found:    817
  > Unique Parent DOIs:      502
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Gates Open Research:
-----------------------------------
  1 version(s):     236 groups
  2 version(s):     222 groups
  3 version(s):     39 groups
  4 version(s):     5 groups

  Average versions per parent: 1.63

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Gates Open Research    501
VeriXiv                  1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/gatesopenres    502



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
gatesopenresearch.org    502


 COMPLETED: GATES OPEN RESEARCH



In [121]:
Gates_parent_df

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
24274,crossref::10.12688/gatesopenres.13123.1,crossref::10.12688/gatesopenres.13123.1,Gates Open Research,10.12688/gatesopenres.13123.1,https://gatesopenresearch.org/articles/4-71/v1,Introducing a drift and diffusion framework fo...,"Lewis, Fraser I; Guga, Godfrey; Mdoe, Paschal;...",2020-06-29,2020,crossref::10.12688/gatesopenres.13123.1; cross...,10.12688/gatesopenres.13123.1; 10.12688/gateso...,Gates Open Research (2),2,2020-06-29,2020-11-26,Gates Open Research,10.12688/gatesopenres.13123.1,10.12688,10.12688,gatesopenres.13123.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
39601,crossref::10.12688/gatesopenres.13457.1,crossref::10.12688/gatesopenres.13457.1,Gates Open Research,10.12688/gatesopenres.13457.1,https://gatesopenresearch.org/articles/5-174/v1,A Randomized controlled trial of the Effect of...,"Harding, Rebecca; Ataide, Ricardo; Mwangi, Mar...",2021-12-07,2021,crossref::10.12688/gatesopenres.13457.1; cross...,10.12688/gatesopenres.13457.1; 10.12688/gateso...,Gates Open Research (2),2,2021-12-07,2022-04-14,Gates Open Research,10.12688/gatesopenres.13457.1,10.12688,10.12688,gatesopenres.13457.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
61653,crossref::10.12688/gatesopenres.12799.2,crossref::10.12688/gatesopenres.12799.2,Gates Open Research,10.12688/gatesopenres.12799.2,https://gatesopenresearch.org/articles/2-13/v2,A sulfur-free peptide mimic of surfactant prot...,"Walther, Frans J.; Gupta, Monik; Gordon, Larry...",2018-07-10,2018,crossref::10.12688/gatesopenres.12799.2,10.12688/gatesopenres.12799.2,Gates Open Research (1),1,2018-07-10,2018-07-10,Gates Open Research,10.12688/gatesopenres.12799.2,10.12688,10.12688,gatesopenres.12799.2,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
68519,crossref::10.12688/gatesopenres.14591.1,crossref::10.12688/gatesopenres.14591.1,Gates Open Research,10.12688/gatesopenres.14591.1,https://gatesopenresearch.org/articles/7-75/v1,Using responsive feedback from routine monitor...,"Meekers, Dominique; Olutola, Olaniyi; Abu Turk...",2023-05-19,2023,crossref::10.12688/gatesopenres.14591.1; cross...,10.12688/gatesopenres.14591.1; 10.12688/gateso...,Gates Open Research (2),2,2023-05-19,2023-11-16,Gates Open Research,10.12688/gatesopenres.14591.1,10.12688,10.12688,gatesopenres.14591.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
75609,crossref::10.12688/gatesopenres.12854.1,crossref::10.12688/gatesopenres.12854.1,Gates Open Research,10.12688/gatesopenres.12854.1,https://gatesopenresearch.org/articles/2-38/v1,Evaluation of a multi-level intervention to im...,"Ingabire, Rosine; Nyombayire, Julien; Hoagland...",2018-08-20,2018,crossref::10.12688/gatesopenres.12854.1; cross...,10.12688/gatesopenres.12854.1; 10.12688/gateso...,Gates Open Research (3),3,2018-08-20,2019-02-04,Gates Open Research,10.12688/gatesopenres.12854.1,10.12688,10.12688,gatesopenres.12854.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7265036,crossref::10.12688/gatesopenres.12954.1,crossref::10.12688/gatesopenres.12954.1,Gates Open Research,10.12688/gatesopenres.12954.1,https://gatesopenresearch.org/articles/3-1466/v1,"Knowledge, perception and experience of sexual...","Sule, Aisha I.; Titiloye, Musibau A.; Arulogun...",2019-05-15,2019,crossref::10.12688/gatesopenres.12

In [122]:
Gates_parent_df[Gates_parent_df['most_recent_version_servers']=='VeriXiv']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
7235707,crossref::10.12688/gatesopenres.16360.1,crossref::10.12688/gatesopenres.16360.1,Gates Open Research,10.12688/gatesopenres.16360.1,https://gatesopenresearch.org/articles/9-65/v1,Driving innovation from discovery to access: M...,"Palmer, Shaun; Clark, Rebecca A.; Connell, Bri...",2025-01-01,2025,crossref::10.12688/gatesopenres.16360.1; cross...,10.12688/gatesopenres.16360.1; 10.12688/verixi...,Gates Open Research (1); VeriXiv (1),2,2025-01-01,2025-05-23,VeriXiv,10.12688/gatesopenres.16360.1,10.12688,10.12688,gatesopenres.16360.1,10.12688/gatesopenres,10.12688/gatesopenres,gatesopenresearch.org,gatesopenresearch.org/articles


In [123]:
Gates_df.drop_duplicates('record_id')

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN..."
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN..."
2,crossref::10.12688/gatesopenres.12816.1,Gates Ope

In [124]:
dup_ti = Gates_df.drop_duplicates('title')

In [125]:
duplicates = dup_ti[dup_ti.duplicated('authors_flat')].sort_values(by='authors_flat')
print(len(duplicates))
duplicates

43


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
128,crossref::10.12688/gatesopenres.12755.1,Gates Open Research,crossref,10.12688/gatesopenres.12755.1,10.12688/gatesopenres.12755.1,https://doi.org/10.12688/gatesopenres.12755.1,https://gatesopenresearch.org/articles/1-6/v1,https://gatesopenresearch.org/articles/1-6/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Evaluating integrated development: are we aski...,None,None,None,en,journal-article,None,journal-article,None,False,2017-11-06,None,2019-10-10,2025-11-16,None,2017-11-06,None,2017-11-06,2017-11-06,2017.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> Emergi...,Background: Emerging global transformations - ...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/1-6/v1/pdf,"Ahner-McHaffie, Tessa W; Guest, Greg; Petruney...",None,None,"[{""affiliation"": [], ""family"": ""Ahner-McHaffie...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation; FHI Foundation,2.0,None,None,None,2,None,None,2,32,"[{""DOI"": ""10.1136/bmj.g5785"", ""article-title"":...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13815.r26084;10.21956/ga...,None,10.12688/gatesopenres.12755.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12755.1"", ""ISSN..."
245,crossref::10.12688/gatesopenres.12870.2,Gates Open Research,crossref,10.12688/gatesopenres.12870.2,10.12688/gatesopenres.12870.2,https://doi.org/10.12688/gatesopenres.12870.2,https://gatesopenresearch.org/articles/2-52/v2,https://gatesopenresearch.org/articles/2-52/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Characterization of fecal sludge as biomass fe...,None,None,None,en,journal-article,None,journal-article,None,False,2020-01-24,None,2020-07-23,2025-06-13,None,2020-01-24,None,2020-01-24,2020-01-24,2020.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns5:p><ns5:bold>Background</ns5:bold>: Transf...,Background : Transformative sanitation technol...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/2-52/v2...,"Barani, Viswa; Hegarty-Craver, Meghan; Rosario...",None,None,"[{""affiliation"": [], ""family"": ""Barani"", ""give...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""4395...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/j.energy.2017.06.010"", ""arti...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,New version,10.21956/gatesopenres.14273

In [126]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN..."
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN..."
2,crossref::10.12688/gatesopenres.12816.1,Gates Ope

In [127]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN..."
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN..."
2,crossref::10.12688/gatesopenres.12816.1,Gates Ope

In [128]:
pattern = "has-preprint"

mask = Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(28, 82)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
788,crossref::10.12688/gatesopenres.16311.1,Gates Open Research,crossref,10.12688/gatesopenres.16311.1,10.12688/gatesopenres.16311.1,https://doi.org/10.12688/gatesopenres.16311.1,https://gatesopenresearch.org/articles/8-143/v1,https://gatesopenresearch.org/articles/8-143/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,An exploration of unusual antimicrobial resist...,None,None,None,en,journal-article,None,journal-article,None,False,2024-12-23,None,2024-12-23,2025-11-21,None,2024-12-23,None,2024-12-23,2024-12-23,2024.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns3:p>\n Typhoid fever is ...,Typhoid fever is a significant public health p...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/8-143/v...,"Zuza, Allan; Wailan, Alexander M.; Anscombe, C...",None,None,"[{""affiliation"": [], ""family"": ""Zuza"", ""given""...",None,None,"[{""DOI"": ""10.13039/100010269"", ""award"": [""2173...",Wellcome Trust; Biotechnology and Biological S...,4.0,None,None,None,0,None,None,0,47,"[{""DOI"": ""10.1016/S1473-3099(18)30685-6"", ""art...","{""has-preprint"": [{""asserted-by"": ""subject"", ""...",10.12688/verixiv.77.2,,False,None,,None,None,,None,None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.16311.1"", ""ISSN..."
799,crossref::10.12688/gatesopenres.16313.1,Gates Open Research,crossref,10.12688/gatesopenres.16313.1,10.12688/gatesopenres.16313.1,https://doi.org/10.12688/gatesopenres.16313.1,https://gatesopenresearch.org/articles/9-1/v1,https://gatesopenresearch.org/articles/9-1/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Automated post-run analysis of arrayed quantit...,None,None,None,en,journal-article,None,journal-article,None,False,2025-01-20,None,2025-01-20,2026-01-05,None,2025-01-20,None,2025-01-20,2025-01-20,2025.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/4.0/,https://creativecommons.org/licenses/by/4.0/,<ns4:p>Background The TaqMan Array Card (TAC) ...,Background The TaqMan Array Card (TAC) is an a...,"[{""URL"": ""https://gatesopenresearch.org/articl...",https://gatesopenresearch.org/articles/9-1/v1/pdf,"Brintz, Ben J.; Operario, Darwin J.; Brown, Da...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-4695-0...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""INV-...",Bill and Melinda Gates Foundation,1.0,None,None,None,1,None,None,1,13,"[{""DOI"": ""10.1136/bmjgh-2022-009548"", ""article...","{""has-preprint"": [{""asserted-by"": ""subject"", ""...",10.12688/verixiv.123.2,,False,None,,None,None,,None,None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DO

In [129]:
Gates_df['has_preprint'].value_counts().reset_index().sort_values(by='has_preprint')

,has_preprint,count
0,,835
18,10.12688/verixiv.1078.1,1
26,10.12688/verixiv.1155.1,1
25,10.12688/verixiv.1161.1,1
15,10.12688/verixiv.1179.2,1
2,10.12688/verixiv.123.2,1
20,10.12688/verixiv.1323.2,1
8,10.12688/verixiv.15.2,1
19,10.12688/verixiv.1752.1,1
17,10.12688/verixiv.1808.1,1


In [130]:
pattern = "has-review"

mask = Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(735, 82)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN..."
2,crossref::10.12688/gatesopenres.12816.1,Gates Open Research,crossref,10.12688/gatesopenres.12816.1,10.12688/gatesopenres.12816.1,https://doi.org/10.12688/gatesopenres.12816.1,https://gatesopenresearch.org/articles/2-24/v1,https://gatesopenresearch.org/articles/2-24/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Funding global health product R&amp;D: the Por...,None,None,None,en,journal-article,None,journal-article,None,False,2018-04-26,None,2018-07-19,2025-02-21,None,2018-04-26,None,2018-04-26,2018-04-26,2018.0,issued_date,None,None,None,https://creativecommons.org/licenses/by/3.0/igo/,https://creativecommons.org/licenses/by/3.0/igo/,<ns4:p><ns4:bold>Background</ns4:bold>: the Po...,Background : the Portfolio-To-Impact (P2I) mod...,"[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Terry, Robert F; Yamey, Gavin; Miyazaki-Krause...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0003-3849-7...",None,None,"[{""award"": [""OPP1151682""], ""award-info"": [{""aw...",Bill and Melinda Gates Foundation; Swiss Agenc...,3.0,None,None,None,2,None,None,2,15,"[{""article-title"": ""Research and Development t...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13883.r26392;10.21956/ga...,None,10.12688/gatesopenres.12816.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,N

In [131]:
pattern = "has-review"

mask = ~Gates_df['relations_json'].str.contains(pattern, regex=False, na=False)
result = Gates_df[mask]
print(result.shape)
result

(128, 82)


,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN..."
6,crossref::10.12688/gatesopenres.12817.2,Gates Open Research,crossref,10.12688/gatesopenres.12817.2,10.12688/gatesopenres.12817.2,https://doi.org/10.12688/gatesopenres.12817.2,https://gatesopenresearch.org/articles/2-23/v2,https://gatesopenresearch.org/articles/2-23/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,Developing new health technologies for neglect...,None,None,None,en,journal-article,None,journal-article,None,False,2018-08-22,None,2018-08-22,2025-11-01,None,2018-08-22,None,2018-08-22,2018-08-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background</ns4:bold>: Fundin...,Background : Funding for neglected disease pro...,"[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Young, Ruth; Bekele, Tewodros; Gunn, Alexander...",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-8787-2...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,4,None,None,4,22,"[{""DOI"": ""10.1016/S0140-6736(13)62105-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12817.1"", ""lab...",10.12688/gatesopenres.12817.3,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12817.2"", ""ISSN..."
8,crossref::10.12688/gatesopenres.12842.2,Gates Open Research,crossref,10.12

In [132]:
Gates_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/gatesopenres.12803.2,Gates Open Research,crossref,10.12688/gatesopenres.12803.2,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.12803.2,https://gatesopenresearch.org/articles/2-11/v2,https://gatesopenresearch.org/articles/2-11/v2,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-07-12,None,2018-07-12,2025-02-21,None,2018-07-12,None,2018-07-12,2018-07-12,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,2,None,None,2,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...",None,,,False,None,,None,New version,,"[{""DOI"": ""10.12688/gatesopenres.12803.1"", ""lab...",None,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.2"", ""ISSN..."
1,crossref::10.12688/gatesopenres.12803.1,Gates Open Research,crossref,10.12688/gatesopenres.12803.1,10.12688/gatesopenres.12803.1,https://doi.org/10.12688/gatesopenres.12803.1,https://gatesopenresearch.org/articles/2-11/v1,https://gatesopenresearch.org/articles/2-11/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,Gates Open Research,None,None,2572-4754,A demographic dividend of the FP2020 Initiativ...,None,None,None,en,journal-article,None,journal-article,None,False,2018-02-22,None,2018-07-12,2025-02-21,None,2018-02-22,None,2018-02-22,2018-02-22,2018.0,issued_date,None,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns4:p><ns4:bold>Background:</ns4:bold> The de...,"Background: The demographic dividend, defined ...","[{""URL"": ""https://gatesopenresearch.org/articl...",None,"Li, Qingfeng; Rimon, Jose G.",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-6390-6...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""OPP1...",Bill and Melinda Gates Foundation,1.0,None,None,None,3,None,None,3,22,"[{""DOI"": ""10.1016/S0140-6736(06)69480-4"", ""art...","{""has-review"": [{""asserted-by"": ""subject"", ""id...",,,False,None,,None,None,10.21956/gatesopenres.13867.r26331;10.21956/ga...,None,10.12688/gatesopenres.12803.2,https://doi.org/10.12688/gatesopenres.crossmar...,issn,49,None,"{""DOI"": ""10.12688/gatesopenres.12803.1"", ""ISSN..."
2,crossref::10.12688/gatesopenres.12816.1,Gates Ope

In [133]:
Gates_df['funders_flat']

0                      Bill and Melinda Gates Foundation
1                      Bill and Melinda Gates Foundation
2      Bill and Melinda Gates Foundation; Swiss Agenc...
3      The Special Programme for Research and Trainin...
4      National Institute of General Medical Sciences...
                             ...                        
858    National Institute of Allergy and Infectious D...
859                                     Gates Foundation
860    Programa grand challenges explorations-brazil:...
861                                                 None
862                                     Gates Foundation
Name: funders_flat, Length: 863, dtype: object

In [134]:
# Optional: Group similar names together
df_funders = Gates_df['funders_flat'].dropna().str.split(';').explode().str.strip()

# # Normalize common variants
# df_funders = df_funders.replace({
#     "Gates Foundation": "Bill and Melinda Gates Foundation"
# })

final_counts = df_funders.value_counts().to_frame()
final_counts.head(10)

,count
funders_flat,
Bill and Melinda Gates Foundation,703
Gates Foundation,52
National Institutes of Health,42
United States Agency for International Development,36
Wellcome Trust,28
Medical Research Council,26
South African Medical Research Council,17
"Department for International Development, UK Government",14
Bill & Melinda Gates Foundation,11


# HAL

In [135]:
# HAL_df, HAL_summary = get_server_data("HAL")

In [136]:
HAL_parent_df, HAL_parent_summary = analyze_parent_data("HAL", full_parent_df)


 CONSOLIDATED ANALYSIS: HAL
  > Total Parent Groups:     1,030,229
  > Total Versions Found:    1,052,746
  > Unique Parent DOIs:      27,498
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HAL:
-----------------------------------
  1 version(s):     1,009,984 groups
  2 version(s):     18,605 groups
  3 version(s):     1,302 groups
  4 version(s):     220 groups
  5 version(s):     58 groups
  6 version(s):     24 groups
  7 version(s):     17 groups
  8 version(s):     4 groups
  9 version(s):     3 groups
  10 version(s):    3 groups
  11 version(s):    3 groups
  12 version(s):    2 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HAL                                         1026395
arXiv                                          1582
ResearchGate      

# HANS Publication PrePrints

In [137]:
HANS_df, HANS_summary = get_server_data("HANS_Publication_PrePrints")


 SERVER ANALYSIS: HANS_PUBLICATION_PREPRINTS
  > Files found:    1
  > Raw records:    75
OK


In [138]:
HANS_parent_df, HANS_parent_summary = analyze_parent_data("HANS Publication PrePrints", full_parent_df)


 CONSOLIDATED ANALYSIS: HANS PUBLICATION PREPRINTS
  > Total Parent Groups:     75
  > Total Versions Found:    75
  > Unique Parent DOIs:      75
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HANS Publication PrePrints:
-----------------------------------
  1 version(s):     75 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HANS Publication PrePrints    75

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12677/hanspreprints    75



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
hanspub.org    75


 COMPLETED: HANS PUBLICATION PREPRINTS



# HRB Open Research

In [139]:
HRB_df, HRB_summary = get_server_data("HRB_Open_Research")


 SERVER ANALYSIS: HRB_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    1012
OK


In [140]:
HRB_parent_df, HRB_parent_summary = analyze_parent_data("HRB Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: HRB OPEN RESEARCH
  > Total Parent Groups:     647
  > Total Versions Found:    1,007
  > Unique Parent DOIs:      647
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for HRB Open Research:
-----------------------------------
  1 version(s):     342 groups
  2 version(s):     254 groups
  3 version(s):     47 groups
  4 version(s):     4 groups

  Average versions per parent: 1.56

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
HRB Open Research    646
SSRN                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/hrbopenres    647



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
hrbopenresearch.org    647


 COMPLETED: HRB OPEN RESEARCH



# Humanities Commons CORE

In [141]:
CORE_df, CORE_summary = get_server_data("Humanities_Commons_CORE")


 SERVER ANALYSIS: HUMANITIES_COMMONS_CORE
  > Files found:    2
  > Raw records:    29584
OK


In [6]:
CORE_parent_df, CORE_parent_summary = analyze_parent_data("Humanities Commons CORE", full_parent_df)


 CONSOLIDATED ANALYSIS: HUMANITIES COMMONS CORE
  > Total Parent Groups:     16,685
  > Total Versions Found:    29,313
  > Unique Parent DOIs:      16,685
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Humanities Commons CORE:
-----------------------------------
  1 version(s):     5,298 groups
  2 version(s):     10,710 groups
  3 version(s):     318 groups
  4 version(s):     288 groups
  5 version(s):     21 groups
  6 version(s):     26 groups
  7 version(s):     4 groups
  8 version(s):     9 groups
  9 version(s):     2 groups
  10 version(s):    3 groups
  12 version(s):    3 groups
  14 version(s):    2 groups
  16 version(s):    1 groups

  Average versions per parent: 1.76

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Humanities Commons CORE            16631
SSRN                                  16
HAL; Humanities Commons CORE           8
Open Science Framework        

# IACR Cryptology ePrint Archive

In [7]:
IACR_df, IACR_summary = get_server_data("IACR_Cryptology_ePrint_Archive")


 SERVER ANALYSIS: IACR_CRYPTOLOGY_EPRINT_ARCHIVE
  > Files found:    1
  > Raw records:    11904
OK


In [8]:
IACR_parent_df, IACR_parent_summary = analyze_parent_data("IACR Cryptology ePrint Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: IACR CRYPTOLOGY EPRINT ARCHIVE
  > Total Parent Groups:     10,430
  > Total Versions Found:    10,679
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for IACR Cryptology ePrint Archive:
-----------------------------------
  1 version(s):     10,196 groups
  2 version(s):     224 groups
  3 version(s):     8 groups
  5 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
IACR Cryptology ePrint Archive                                   10307
arXiv                                                              110
HAL                                                                  5
ResearchGate                                                         3
SSRN                                                                 2
IACR Cryptology ePrint Archiv

# INA-Rxiv

In [9]:
INA_df, INA_summary = get_server_data("INA-Rxiv")


 SERVER ANALYSIS: INA-RXIV
  > Files found:    1
  > Raw records:    17837
OK


In [10]:
INA_parent_df, INA_parent_summary = analyze_parent_data("INA-Rxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: INA-RXIV
  > Total Parent Groups:     15,055
  > Total Versions Found:    20,750
  > Unique Parent DOIs:      15,055
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for INA-Rxiv:
-----------------------------------
  1 version(s):     11,033 groups
  2 version(s):     3,072 groups
  3 version(s):     711 groups
  4 version(s):     134 groups
  5 version(s):     42 groups
  6 version(s):     23 groups
  7 version(s):     7 groups
  8 version(s):     6 groups
  9 version(s):     5 groups
  10 version(s):    5 groups
  12 version(s):    3 groups
  13 version(s):    2 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    2 groups
  21 version(s):    1 groups
  22 version(s):    1 groups
  27 version(s):    1 groups
  50 version(s):    1 groups
  56 version(s):    1 groups

  Average versions per parent: 1.38

 MOST RECENT DESTINATIONS

# IndiaRxiv

In [11]:
IndiaRxiv_df, IndiaRxiv_summary = get_server_data("IndiaRxiv")


 SERVER ANALYSIS: INDIARXIV
  > Files found:    2
  > Raw records:    142
OK


In [12]:
IndiaRxiv_parent_df, IndiaRxiv_parent_summary = analyze_parent_data("IndiaRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: INDIARXIV
  > Total Parent Groups:     103
  > Total Versions Found:    110
  > Unique Parent DOIs:      103
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for IndiaRxiv:
-----------------------------------
  1 version(s):     96 groups
  2 version(s):     7 groups

  Average versions per parent: 1.07

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
IndiaRxiv          98
ChemRxiv            1
arXiv               1
engrXiv             1
Research Square     1
Thesis Commons      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.35543/osf          97
10.35543/indiarxiv     6



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io             97
ops.iihr.res.in     6


 COMPLETED: INDIARXIV



# JMIR Preprints

In [13]:
JMIR_df, JMIR_summary = get_server_data("JMIR_Preprints")


 SERVER ANALYSIS: JMIR_PREPRINTS
  > Files found:    1
  > Raw records:    37631
OK


In [14]:
JMIR_parent_df, JMIR_parent_summary = analyze_parent_data("JMIR Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: JMIR PREPRINTS
  > Total Parent Groups:     35,807
  > Total Versions Found:    37,218
  > Unique Parent DOIs:      35,807
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for JMIR Preprints:
-----------------------------------
  1 version(s):     34,506 groups
  2 version(s):     1,234 groups
  3 version(s):     59 groups
  4 version(s):     5 groups
  5 version(s):     2 groups
  37 version(s):    1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
JMIR Preprints              35450
Research Square               107
medRxiv                        86
arXiv                          53
SSRN                           35
Preprints.org                  16
PsyArXiv                       13
JMIR Preprints; PsyArXiv        7
JMIR Preprints; arXiv           5
bioRxiv                         4

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKE

# Jxiv

In [15]:
Jxiv_df, Jxiv_summary = get_server_data("Jxiv")


 SERVER ANALYSIS: JXIV
  > Files found:    1
  > Raw records:    902
OK


In [16]:
Jxiv_parent_df, Jxiv_parent_summary = analyze_parent_data("Jxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: JXIV
  > Total Parent Groups:     757
  > Total Versions Found:    772
  > Unique Parent DOIs:      757
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Jxiv:
-----------------------------------
  1 version(s):     746 groups
  2 version(s):     10 groups
  6 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Jxiv               753
SSRN                 2
Research Square      2

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.51094/jxiv    757



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
jxiv.jst.go.jp    757


 COMPLETED: JXIV



# Keldysh Institute Preprints

In [17]:
Keldysh_df, Keldysh_summary = get_server_data("Keldysh_Institute_Preprints")


 SERVER ANALYSIS: KELDYSH_INSTITUTE_PREPRINTS
  > Files found:    1
  > Raw records:    1258
OK


In [18]:
Keldysh_parent_df, Keldysh_parent_summary = analyze_parent_data("Keldysh Institute Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: KELDYSH INSTITUTE PREPRINTS
  > Total Parent Groups:     1,193
  > Total Versions Found:    1,257
  > Unique Parent DOIs:      1,193
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Keldysh Institute Preprints:
-----------------------------------
  1 version(s):     1,135 groups
  2 version(s):     54 groups
  3 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Keldysh Institute Preprints    1185
arXiv                             8

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.20948/prepr-       1192
10.20948/preprints       1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
keldysh.ru            1192
library.keldysh.ru       1


 COMPLETED: KELDYSH INSTITUTE PREPRINTS



# LatArXiv

In [19]:
LatArXiv_df, LatArXiv_summary = get_server_data("LatArXiv")


 SERVER ANALYSIS: LATARXIV
  > Files found:    1
  > Raw records:    125
OK


In [20]:
LatArXiv_parent_df, LatArXiv_parent_summary = analyze_parent_data("LatArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: LATARXIV
  > Total Parent Groups:     115
  > Total Versions Found:    118
  > Unique Parent DOIs:      115
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LatArXiv:
-----------------------------------
  1 version(s):     112 groups
  2 version(s):     3 groups

  Average versions per parent: 1.03

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LatArXiv            114
SciELO Preprints      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.62059/latarxiv    115



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprints.latarxiv.org    115


 COMPLETED: LATARXIV



# Law Archive

In [21]:
Law_Archive_df, Law_Archive_summary = get_server_data("Law_Archive")


 SERVER ANALYSIS: LAW_ARCHIVE
  > Files found:    1
  > Raw records:    1808
OK


In [22]:
Law_Archive_parent_df, Law_Archive_parent_summary = analyze_parent_data("Law Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: LAW ARCHIVE
  > Total Parent Groups:     1,344
  > Total Versions Found:    2,212
  > Unique Parent DOIs:      1,344
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Law Archive:
-----------------------------------
  1 version(s):     634 groups
  2 version(s):     578 groups
  3 version(s):     124 groups
  4 version(s):     4 groups
  5 version(s):     2 groups
  11 version(s):    1 groups
  13 version(s):    1 groups

  Average versions per parent: 1.65

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Law Archive                            681
Law Archive; Open Science Framework    471
Open Science Framework                 144
SSRN                                    32
arXiv                                    5
EdArXiv                                  3
CrimRxiv                                 2
Law Archive; arXiv                       2
EdArXiv; Law Archiv

# LIS Scholarship Archive

In [23]:
LIS_df, LIS_summary = get_server_data("LIS_Scholarship_Archive")


 SERVER ANALYSIS: LIS_SCHOLARSHIP_ARCHIVE
  > Files found:    1
  > Raw records:    397
OK


In [24]:
LIS_parent_df, LIS_parent_summary = analyze_parent_data("LIS Scholarship Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: LIS SCHOLARSHIP ARCHIVE
  > Total Parent Groups:     328
  > Total Versions Found:    488
  > Unique Parent DOIs:      328
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LIS Scholarship Archive:
-----------------------------------
  1 version(s):     201 groups
  2 version(s):     97 groups
  3 version(s):     27 groups
  4 version(s):     3 groups

  Average versions per parent: 1.49

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LIS Scholarship Archive                            217
LIS Scholarship Archive; Open Science Framework     96
Open Science Framework                               3
Thesis Commons                                       3
Frenxiv                                              2
LIS Scholarship Archive; ResearchGate                1
SSRN                                                 1
Zenodo                                           

# LSE Research Online Documents on Economics

In [25]:
LSE_df, LSE_summary = get_server_data("LSE_Research_Online_Documents_on_Economics")


 SERVER ANALYSIS: LSE_RESEARCH_ONLINE_DOCUMENTS_ON_ECONOMICS
  > Files found:    1
  > Raw records:    119
OK


In [26]:
LSE_parent_df, LSE_parent_summary = analyze_parent_data("LSE Research Online Documents on Economics", full_parent_df)


 CONSOLIDATED ANALYSIS: LSE RESEARCH ONLINE DOCUMENTS ON ECONOMICS
  > Total Parent Groups:     110
  > Total Versions Found:    144
  > Unique Parent DOIs:      11
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for LSE Research Online Documents on Economics:
-----------------------------------
  1 version(s):     86 groups
  2 version(s):     15 groups
  3 version(s):     8 groups
  4 version(s):     1 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
LSE Research Online Documents on Economics                        86
SSRN                                                              14
RePEc: Research Papers in Economics                                9
EconStor Preprints; LSE Research Online Documents on Economics     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>          99
10.508

# MarXiv

In [27]:
MarXiv_df, MarXiv_summary = get_server_data("MarXiv")


 SERVER ANALYSIS: MARXIV
  > Files found:    1
  > Raw records:    508
OK


In [28]:
MarXiv_parent_df, MarXiv_parent_summary = analyze_parent_data("MarXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MARXIV
  > Total Parent Groups:     509
  > Total Versions Found:    672
  > Unique Parent DOIs:      509
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MarXiv:
-----------------------------------
  1 version(s):     360 groups
  2 version(s):     135 groups
  3 version(s):     14 groups

  Average versions per parent: 1.32

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MarXiv                            489
MarXiv; Open Science Framework     17
INA-Rxiv                            1
Open Science Framework              1
EdArXiv                             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31230/osf    476
10.17605/osf     33



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io        476
marxiv.org     33


 COMPLETED: MARXIV



# MediArXiv

In [29]:
MediArXiv_df, MediArXiv_summary = get_server_data("MediArXiv")


 SERVER ANALYSIS: MEDIARXIV
  > Files found:    1
  > Raw records:    309
OK


In [30]:
MediArXiv_parent_df, MediArXiv_parent_summary = analyze_parent_data("MediArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MEDIARXIV
  > Total Parent Groups:     277
  > Total Versions Found:    299
  > Unique Parent DOIs:      277
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MediArXiv:
-----------------------------------
  1 version(s):     260 groups
  2 version(s):     13 groups
  3 version(s):     3 groups
  4 version(s):     1 groups

  Average versions per parent: 1.08

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MediArXiv                  270
SSRN                         2
Zenodo                       1
SocArXiv                     1
Humanities Commons CORE      1
arXiv                        1
APSA Preprints               1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.33767/osf    274
10.31219/osf      3



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    277


 COM

# medRxiv

In [31]:
medRxiv_df, medRxiv_summary = get_server_data("medRxiv")


 SERVER ANALYSIS: MEDRXIV
  > Files found:    1
  > Raw records:    75743
OK


In [32]:
medRxiv_parent_df, medRxiv_parent_summary = analyze_parent_data("medRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MEDRXIV
  > Total Parent Groups:     74,055
  > Total Versions Found:    78,040
  > Unique Parent DOIs:      74,055
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for medRxiv:
-----------------------------------
  1 version(s):     70,734 groups
  2 version(s):     2,928 groups
  3 version(s):     246 groups
  4 version(s):     38 groups
  5 version(s):     94 groups
  6 version(s):     15 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
medRxiv                   70925
Research Square            1194
SSRN                        693
eLife                       388
JMIR Preprints              305
Authorea Inc.                94
arXiv                        62
Wellcome Open Research       53
Preprints.org                39
F1000Research                38

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
----------------------------

# MetaArXiv

In [33]:
MetaArXiv_df, MetaArXiv_summary = get_server_data("MetaArXiv")


 SERVER ANALYSIS: METAARXIV
  > Files found:    1
  > Raw records:    880
OK


In [34]:
MetaArXiv_parent_df, MetaArXiv_parent_summary = analyze_parent_data("MetaArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: METAARXIV
  > Total Parent Groups:     705
  > Total Versions Found:    907
  > Unique Parent DOIs:      705
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MetaArXiv:
-----------------------------------
  1 version(s):     547 groups
  2 version(s):     124 groups
  3 version(s):     26 groups
  4 version(s):     6 groups
  5 version(s):     2 groups

  Average versions per parent: 1.29

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MetaArXiv                            633
SSRN                                  14
MetaArXiv; Open Science Framework     10
Open Science Framework                10
SocArXiv                               6
eLife                                  5
Frenxiv                                4
Research Square                        3
Thesis Commons                         3
medRxiv                                2

 TOP VALUES FOR: DOI_

# MindRxiv

In [35]:
MindRxiv_df, MindRxiv_summary = get_server_data("MindRxiv")


 SERVER ANALYSIS: MINDRXIV
  > Files found:    1
  > Raw records:    335
OK


In [36]:
MindRxiv_parent_df, MindRxiv_parent_summary = analyze_parent_data("MindRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: MINDRXIV
  > Total Parent Groups:     281
  > Total Versions Found:    402
  > Unique Parent DOIs:      281
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MindRxiv:
-----------------------------------
  1 version(s):     198 groups
  2 version(s):     45 groups
  3 version(s):     38 groups

  Average versions per parent: 1.43

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MindRxiv                            245
MindRxiv; Open Science Framework     32
MindRxiv; PsyArXiv                    1
SSRN                                  1
EdArXiv; MindRxiv                     1
Open Science Framework                1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31231/osf    281



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    281


 COMPLETED: MINDRXIV



# MNI Open Research

In [37]:
MNI_df, MNI_summary = get_server_data("MNI_Open_Research")


 SERVER ANALYSIS: MNI_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    20
OK


In [38]:
MNI_parent_df, MNI_parent_summary = analyze_parent_data("MNI Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: MNI OPEN RESEARCH
  > Total Parent Groups:     13
  > Total Versions Found:    19
  > Unique Parent DOIs:      13
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for MNI Open Research:
-----------------------------------
  1 version(s):     7 groups
  2 version(s):     6 groups

  Average versions per parent: 1.46

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
MNI Open Research    13

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/mniopenres    13



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
mniopenresearch.org    13


 COMPLETED: MNI OPEN RESEARCH



# Munich Personal RePEc Archive

In [39]:
Munich_df, Munich_summary = get_server_data("Munich_Personal_RePEc_Archive")

/tmp/ipykernel_3588/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: MUNICH_PERSONAL_REPEC_ARCHIVE
  > Files found:    7
  > Raw records:    68692
OK


In [40]:
Munich_parent_df, Munich_parent_summary = analyze_parent_data("Munich Personal RePEc Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: MUNICH PERSONAL REPEC ARCHIVE
  > Total Parent Groups:     66,342
  > Total Versions Found:    68,859
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Munich Personal RePEc Archive:
-----------------------------------
  1 version(s):     63,971 groups
  2 version(s):     2,255 groups
  3 version(s):     100 groups
  4 version(s):     10 groups
  5 version(s):     3 groups
  7 version(s):     1 groups
  8 version(s):     2 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Munich Personal RePEc Archive                                         64032
SSRN                                                                   1784
HAL; Munich Personal RePEc Archive                                      219
arXiv                                                                    90
RePEc: Research Pap

# National Bureau of Economic Research

In [41]:
National_Bureau_df, National_Bureau_summary = get_server_data("National_Bureau_of_Economic_Research")


 SERVER ANALYSIS: NATIONAL_BUREAU_OF_ECONOMIC_RESEARCH
  > Files found:    5
  > Raw records:    1856
OK


In [42]:
National_Bureau_parent_df, National_Bureau_parent_summary = analyze_parent_data("National Bureau of Economic Research", full_parent_df)


 CONSOLIDATED ANALYSIS: NATIONAL BUREAU OF ECONOMIC RESEARCH
  > Total Parent Groups:     1,663
  > Total Versions Found:    2,265
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for National Bureau of Economic Research:
-----------------------------------
  1 version(s):     1,131 groups
  2 version(s):     474 groups
  3 version(s):     48 groups
  4 version(s):     8 groups
  5 version(s):     2 groups

  Average versions per parent: 1.36

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
National Bureau of Economic Research                                         1146
SSRN                                                                          381
RePEc: Research Papers in Economics                                            91
National Bureau of Economic Research; RePEc: Research Papers in Economics      28
AgEcon Search                                 

# Nature Precedings

In [43]:
Nature_df, Nature_summary = get_server_data("Nature_Precedings")


 SERVER ANALYSIS: NATURE_PRECEDINGS
  > Files found:    1
  > Raw records:    5210
OK


In [44]:
Nature_parent_df, Nature_parent_summary = analyze_parent_data("Nature Precedings", full_parent_df)


 CONSOLIDATED ANALYSIS: NATURE PRECEDINGS
  > Total Parent Groups:     3,153
  > Total Versions Found:    5,187
  > Unique Parent DOIs:      3,153
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Nature Precedings:
-----------------------------------
  1 version(s):     1,394 groups
  2 version(s):     1,576 groups
  3 version(s):     120 groups
  4 version(s):     51 groups
  5 version(s):     7 groups
  6 version(s):     2 groups
  10 version(s):    3 groups

  Average versions per parent: 1.65

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Nature Precedings                         3111
arXiv                                       23
Nature Precedings; arXiv                     8
SSRN                                         3
F1000Research                                2
bioRxiv                                      2
DSpace@MIT                                   1
viXra            

# NewAddictionsX

In [45]:
NewAddictionsX_df, NewAddictionsX_summary = get_server_data("NewAddictionsX")


 SERVER ANALYSIS: NEWADDICTIONSX
  > Files found:    1
  > Raw records:    7
OK


In [46]:
NewAddictionsX_parent_df, NewAddictionsX_parent_summary = analyze_parent_data("NewAddictionsX", full_parent_df)


 CONSOLIDATED ANALYSIS: NEWADDICTIONSX
  > Total Parent Groups:     4
  > Total Versions Found:    7
  > Unique Parent DOIs:      4
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for NewAddictionsX:
-----------------------------------
  1 version(s):     2 groups
  2 version(s):     1 groups
  3 version(s):     1 groups

  Average versions per parent: 1.75

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
NewAddictionsX    4

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31219/osf    4



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    4


 COMPLETED: NEWADDICTIONSX



# NutriXiv

In [47]:
NutriXiv_df, NutriXiv_summary = get_server_data("NutriXiv")


 SERVER ANALYSIS: NUTRIXIV
  > Files found:    1
  > Raw records:    94
OK


In [48]:
NutriXiv_parent_df, NutriXiv_parent_summary = analyze_parent_data("NutriXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: NUTRIXIV
  > Total Parent Groups:     83
  > Total Versions Found:    111
  > Unique Parent DOIs:      83
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for NutriXiv:
-----------------------------------
  1 version(s):     63 groups
  2 version(s):     16 groups
  3 version(s):     2 groups
  4 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
NutriXiv                            66
NutriXiv; Open Science Framework    14
SSRN                                 1
IndiaRxiv                            1
Research Square                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31232/osf    83



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    83


 COMPLETED: NUTRIXIV



# Open Research Africa

In [49]:
openra_df, openra_summary = get_server_data("Open_Research_Africa")


 SERVER ANALYSIS: OPEN_RESEARCH_AFRICA
  > Files found:    1
  > Raw records:    288
OK


In [50]:
openra_parent_df, openra_parent_summary = analyze_parent_data("Open Research Africa", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN RESEARCH AFRICA
  > Total Parent Groups:     180
  > Total Versions Found:    276
  > Unique Parent DOIs:      180
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Research Africa:
-----------------------------------
  1 version(s):     99 groups
  2 version(s):     66 groups
  3 version(s):     15 groups

  Average versions per parent: 1.53

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Open Research Africa    179
F1000Research             1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/aasopenres       134
10.12688/openresafrica     46



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
aasopenresearch.org       112
openresearchafrica.org     68


 COMPLETED: OPEN RESEARCH AFRICA



# Open Research Europe

In [51]:
openre_df, openre_summary = get_server_data("Open_Research_Europe")


 SERVER ANALYSIS: OPEN_RESEARCH_EUROPE
  > Files found:    1
  > Raw records:    1877
OK


In [52]:
openre_parent_df, openre_parent_summary = analyze_parent_data("Open Research Europe", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN RESEARCH EUROPE
  > Total Parent Groups:     1,115
  > Total Versions Found:    1,776
  > Unique Parent DOIs:      1,115
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Research Europe:
-----------------------------------
  1 version(s):     563 groups
  2 version(s):     455 groups
  3 version(s):     86 groups
  4 version(s):     10 groups
  5 version(s):     1 groups

  Average versions per parent: 1.59

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Open Research Europe    1110
arXiv                      3
SSRN                       1
Zenodo                     1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/openreseurope    1115



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
open-research-europe.ec.europa.eu    1115


 COMPLETED: OPEN RESEARCH 

# Open Science Framework

In [53]:
osf_df, osf_summary = get_server_data("Open_Science_Framework")

/tmp/ipykernel_3588/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)



 SERVER ANALYSIS: OPEN_SCIENCE_FRAMEWORK
  > Files found:    3
  > Raw records:    119481
OK


In [54]:
osf_parent_df, osf_parent_summary = analyze_parent_data("Open Science Framework", full_parent_df)


 CONSOLIDATED ANALYSIS: OPEN SCIENCE FRAMEWORK
  > Total Parent Groups:     94,335
  > Total Versions Found:    110,556
  > Unique Parent DOIs:      94,335
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Open Science Framework:
-----------------------------------
  1 version(s):     83,245 groups
  2 version(s):     8,579 groups
  3 version(s):     1,634 groups
  4 version(s):     516 groups
  5 version(s):     158 groups
  6 version(s):     69 groups
  7 version(s):     42 groups
  8 version(s):     26 groups
  9 version(s):     20 groups
  10 version(s):    8 groups
  11 version(s):    7 groups
  12 version(s):    4 groups
  13 version(s):    3 groups
  14 version(s):    4 groups
  15 version(s):    3 groups
  17 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    1 groups
  22 version(s):    1 groups
  25 version(s):    1 groups
  28 version(s):    1 groups
  30 version(s):    1 groups

# Organic Eprints

In [55]:
Organic_Eprints_df, Organic_Eprints_summary = get_server_data("Organic_Eprints")


 SERVER ANALYSIS: ORGANIC_EPRINTS
  > Files found:    1
  > Raw records:    13983
OK


In [56]:
Organic_parent_df, Organic_parent_summary = analyze_parent_data("Organic Eprints", full_parent_df)


 CONSOLIDATED ANALYSIS: ORGANIC EPRINTS
  > Total Parent Groups:     13,510
  > Total Versions Found:    13,731
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Organic Eprints:
-----------------------------------
  1 version(s):     13,304 groups
  2 version(s):     192 groups
  3 version(s):     13 groups
  4 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Organic Eprints         13471
AgEcon Search              22
HAL                        10
HAL; Organic Eprints        7

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    13510



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None             12079
orgprints.org     1406
                    24
bioaktuell.ch        1


 COMPLETED: ORGANIC EPRINTS


# Oroboros Instruments

In [57]:
Oroboros_Instruments_df, Oroboros_Instruments_summary = get_server_data("Oroboros_Instruments")


 SERVER ANALYSIS: OROBOROS_INSTRUMENTS
  > Files found:    2
  > Raw records:    95
OK


/tmp/ipykernel_3588/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [58]:
Oroboros_parent_df, Oroboros_parent_summary = analyze_parent_data("Oroboros Instruments", full_parent_df)


 CONSOLIDATED ANALYSIS: OROBOROS INSTRUMENTS
  > Total Parent Groups:     68
  > Total Versions Found:    89
  > Unique Parent DOIs:      68
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Oroboros Instruments:
-----------------------------------
  1 version(s):     52 groups
  2 version(s):     14 groups
  3 version(s):     1 groups
  6 version(s):     1 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Oroboros Instruments    68

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.26124/mitofit    45
10.26124/bec        15
10.26124/becprep     8



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
wiki.oroboros.at                    30
bioenergetics-communications.org    23
mitofit.org                         15


 COMPLETED: OROBOROS INSTRUMENTS



# PaleorXiv

In [59]:
PaleorXiv_df, PaleorXiv_summary = get_server_data("PaleorXiv")


 SERVER ANALYSIS: PALEORXIV
  > Files found:    1
  > Raw records:    287
OK


In [60]:
PaleorXiv_parent_df, PaleorXiv_parent_summary = analyze_parent_data("PaleorXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: PALEORXIV
  > Total Parent Groups:     243
  > Total Versions Found:    362
  > Unique Parent DOIs:      243
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PaleorXiv:
-----------------------------------
  1 version(s):     150 groups
  2 version(s):     72 groups
  3 version(s):     17 groups
  4 version(s):     3 groups
  5 version(s):     1 groups

  Average versions per parent: 1.49

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PaleorXiv                            191
Open Science Framework; PaleorXiv     51
Preprints.org                          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31233/osf    243



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
osf.io    243


 COMPLETED: PALEORXIV



# PeerJ Preprints

In [61]:
PeerJ_df, PeerJ_summary = get_server_data("PeerJ_Preprints")


 SERVER ANALYSIS: PEERJ_PREPRINTS
  > Files found:    1
  > Raw records:    6446
OK


In [62]:
PeerJ_parent_df, PeerJ_parent_summary = analyze_parent_data("PeerJ Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: PEERJ PREPRINTS
  > Total Parent Groups:     5,141
  > Total Versions Found:    6,440
  > Unique Parent DOIs:      5,141
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PeerJ Preprints:
-----------------------------------
  1 version(s):     4,216 groups
  2 version(s):     686 groups
  3 version(s):     161 groups
  4 version(s):     46 groups
  5 version(s):     22 groups
  6 version(s):     5 groups
  7 version(s):     3 groups
  12 version(s):    2 groups

  Average versions per parent: 1.25

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PeerJ Preprints           5082
arXiv                       20
F1000Research               11
Open Science Framework       3
bioRxiv                      3
Zenodo                       3
MarXiv                       2
ResearchGate                 2
HAL                          2
PeerJ Preprints; arXiv       2

 TOP VALUES

# PhilSci-Archive

In [63]:
PhilSci_df, PhilSci_summary = get_server_data("PhilSci-Archive")


 SERVER ANALYSIS: PHILSCI-ARCHIVE
  > Files found:    1
  > Raw records:    2362
OK


In [64]:
PhilSci_parent_df, PhilSci_parent_summary = analyze_parent_data("PhilSci-Archive", full_parent_df)


 CONSOLIDATED ANALYSIS: PHILSCI-ARCHIVE
  > Total Parent Groups:     2,129
  > Total Versions Found:    2,165
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PhilSci-Archive:
-----------------------------------
  1 version(s):     2,097 groups
  2 version(s):     29 groups
  3 version(s):     2 groups
  4 version(s):     1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PhilSci-Archive    2108
arXiv                18
bioRxiv               1
ResearchGate          1
HAL                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
<NA>    2129



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
None                        1645
philsci-archive.pitt.edu     484


 COMPLETED: PHILSCI-ARCHIVE



# PoolText

In [65]:
PoolText_df, PoolText_summary = get_server_data("PoolText")


 SERVER ANALYSIS: POOLTEXT
  > Files found:    1
  > Raw records:    79
OK


In [66]:
PoolText_parent_df, PoolText_parent_summary = analyze_parent_data("PoolText", full_parent_df)


 CONSOLIDATED ANALYSIS: POOLTEXT
  > Total Parent Groups:     79
  > Total Versions Found:    79
  > Unique Parent DOIs:      79
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PoolText:
-----------------------------------
  1 version(s):     79 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PoolText    79

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.31923/pooltext-preprint-    78
10.31923/55                     1



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
content.pooltext.com    79


 COMPLETED: POOLTEXT



# prepare@u

In [67]:
prepare_df, prepare_summary = get_server_data("prepare@u")


 SERVER ANALYSIS: PREPARE@U
  > Files found:    1
  > Raw records:    227
OK


In [68]:
prepare_parent_df, prepare_parent_summary = analyze_parent_data("prepare@u", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPARE@U
  > Total Parent Groups:     224
  > Total Versions Found:    227
  > Unique Parent DOIs:      224
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for prepare@u:
-----------------------------------
  1 version(s):     221 groups
  2 version(s):     3 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
prepare@u          223
Research Square      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.36375/prepare    224



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
preprint.prepare.org.in    220
prepare.org.in               2
prepare.enggtalks.com        2


 COMPLETED: PREPARE@U



# Preprints.org

In [69]:
Preprints_df, Preprints_summary = get_server_data("Preprints.org")


 SERVER ANALYSIS: PREPRINTS.ORG
  > Files found:    1
  > Raw records:    115815
OK


In [70]:
Preprints_parent_df, Preprints_parent_summary = analyze_parent_data("Preprints.org", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPRINTS.ORG
  > Total Parent Groups:     103,649
  > Total Versions Found:    115,070
  > Unique Parent DOIs:      103,649
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Preprints.org:
-----------------------------------
  1 version(s):     95,681 groups
  2 version(s):     6,404 groups
  3 version(s):     1,011 groups
  4 version(s):     276 groups
  5 version(s):     105 groups
  6 version(s):     53 groups
  7 version(s):     28 groups
  8 version(s):     16 groups
  9 version(s):     14 groups
  10 version(s):    8 groups
  11 version(s):    12 groups
  12 version(s):    9 groups
  13 version(s):    4 groups
  14 version(s):    3 groups
  15 version(s):    2 groups
  16 version(s):    1 groups
  17 version(s):    3 groups
  18 version(s):    2 groups
  20 version(s):    1 groups
  21 version(s):    2 groups
  22 version(s):    1 groups
  23 version(s):    5 groups
  26 version(s):    3 groups
  36 version(s

# PREPRINTS.RU

In [71]:
PREPRINTS_df, PREPRINTS_summary = get_server_data("PREPRINTS.RU")


 SERVER ANALYSIS: PREPRINTS.RU
  > Files found:    1
  > Raw records:    1415
OK


In [72]:
PREPRINTS_parent_df, PREPRINTS_parent_summary = analyze_parent_data("PREPRINTS.RU", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPRINTS.RU
  > Total Parent Groups:     1,346
  > Total Versions Found:    1,415
  > Unique Parent DOIs:      1,346
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PREPRINTS.RU:
-----------------------------------
  1 version(s):     1,295 groups
  2 version(s):     42 groups
  3 version(s):     4 groups
  4 version(s):     2 groups
  5 version(s):     2 groups
  6 version(s):     1 groups

  Average versions per parent: 1.05

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PREPRINTS.RU                  1334
PREPRINTS.RU; Zenodo             3
SSRN                             2
PREPRINTS.RU; ResearchGate       1
engrXiv                          1
AfricArXiv; PREPRINTS.RU         1
Advance                          1
Open Science Framework           1
Covid-19 Preprints               1
Cambridge Open Engage            1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN


# Prepublicaciones OpenCiencia

In [73]:
OpenCiencia_df, OpenCiencia_summary = get_server_data("Prepublicaciones_OpenCiencia")


 SERVER ANALYSIS: PREPUBLICACIONES_OPENCIENCIA
  > Files found:    1
  > Raw records:    8
OK


In [74]:
OpenCiencia_parent_df, OpenCiencia_parent_summary = analyze_parent_data("Prepublicaciones OpenCiencia", full_parent_df)


 CONSOLIDATED ANALYSIS: PREPUBLICACIONES OPENCIENCIA
  > Total Parent Groups:     7
  > Total Versions Found:    8
  > Unique Parent DOIs:      7
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Prepublicaciones OpenCiencia:
-----------------------------------
  1 version(s):     6 groups
  2 version(s):     1 groups

  Average versions per parent: 1.14

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Prepublicaciones OpenCiencia    7

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.47073/preprints    7



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
prepublicaciones.org    7


 COMPLETED: PREPUBLICACIONES OPENCIENCIA



# PropylaeumDok

In [75]:
PropylaeumDok_df, PropylaeumDok_summary = get_server_data("PropylaeumDok")


 SERVER ANALYSIS: PROPYLAEUMDOK
  > Files found:    2
  > Raw records:    6750
OK


In [76]:
PropylaeumDok_parent_df, PropylaeumDok_parent_summary = analyze_parent_data("PropylaeumDok", full_parent_df)


 CONSOLIDATED ANALYSIS: PROPYLAEUMDOK
  > Total Parent Groups:     6,664
  > Total Versions Found:    6,748
  > Unique Parent DOIs:      6,664
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PropylaeumDok:
-----------------------------------
  1 version(s):     6,608 groups
  2 version(s):     46 groups
  3 version(s):     5 groups
  4 version(s):     2 groups
  5 version(s):     1 groups
  6 version(s):     1 groups
  14 version(s):    1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
PropylaeumDok    6664

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.11588/propylaeumdok    6664



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
archiv.ub.uni-heidelberg.de    6664


 COMPLETED: PROPYLAEUMDOK



In [77]:
PropylaeumDok_df[PropylaeumDok_df['type_backend_raw']=='Film']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
757,datacite::10.11588/propylaeumdok.00000379,PropylaeumDok,datacite,10.11588/propylaeumdok.00000379,10.11588/propylaeumdok.00000379,https://doi.org/10.11588/propylaeumdok.00000379,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,Experimente zur Keramikherstellung: Bau und Te...,None,None,None,None,Film,Video,Film,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2009.0,published_year,None,None,None,None,None,[],None,None,None,"Hampe, Roland; Winter, Adam",None,None,"[{""affiliation"": [], ""familyName"": ""Hampe"", ""g...",[],None,[],None,None,"[{""subject"": ""Plastic arts Sculpture""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content..."
758,datacite::10.11588/propylaeumdok.00000380,PropylaeumDok,datacite,10.11588/propylaeumdok.00000380,10.11588/propylaeumdok.00000380,https://doi.org/10.11588/propylaeumdok.00000380,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,Experimente zur Keramikherstellung: Becher aus...,None,None,None,None,Film,Video,Film,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2009.0,published_year,None,None,None,None,None,[],None,None,None,"Hampe, Roland; Winter, Adam",None,None,"[{""affiliation"": [], ""familyName"": ""Hampe"", ""g...",[],None,[],None,None,"[{""subject"": ""Plastic arts Sculpture""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content..."
763,datacite::10.11588/propylaeumdok.00000381,PropylaeumDok,datacite,10.11588/propylaeumdok.00000381,10.11588/propylaeumdok.00000381,https://doi.org/10.11588/propylaeumdok.00000381,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,Experimente zur Keramikherstellung: Heidelberg...,None,None,None,None,Film,Video,Film,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2009.0,published_year,None,None,None,None,None,[],None,None,None,"Hampe, Roland; Winter, Adam",None,None,"[{""affiliation"": [], ""familyName"": ""Hampe"", ""g...",[],None,[],None,None,"[{""subject"": ""Plastic arts Sculpture""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefi

In [78]:
PropylaeumDok_df[PropylaeumDok_df['type_backend_raw']=='Collection']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
523,datacite::10.11588/propylaeumdok.00000139,PropylaeumDok,datacite,10.11588/propylaeumdok.00000139,10.11588/propylaeumdok.00000139,https://doi.org/10.11588/propylaeumdok.00000139,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,OCCIDENT & ORIENT: Newsletter of the German Pr...,None,None,None,None,Collection,Periodical,Collection,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2008.0,published_year,None,None,None,None,None,[],None,None,None,Deutsches Evangelisches Institut Für Altertums...,None,None,"[{""affiliation"": [], ""name"": ""Deutsches Evange...",[],None,[],None,None,"[{""subject"": ""Palästina, Israel (Altertum)""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content..."
524,datacite::10.11588/propylaeumdok.00000140,PropylaeumDok,datacite,10.11588/propylaeumdok.00000140,10.11588/propylaeumdok.00000140,https://doi.org/10.11588/propylaeumdok.00000140,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,OCCIDENT & ORIENT: Newsletter of the German Pr...,None,None,None,None,Collection,Periodical,Collection,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2008.0,published_year,None,None,None,None,None,[],None,None,None,Deutsches Evangelisches Institut Für Altertums...,None,None,"[{""affiliation"": [], ""name"": ""Deutsches Evange...",[],None,[],None,None,"[{""subject"": ""Palästina, Israel (Altertum)""}]",None,None,0,0,None,None,0,None,[],,,False,None,,None,None,,None,None,None,client_id/doi_prefix_first_token,8,"{""client"": {""data"": {""id"": ""gesis.ubhd"", ""type...","{""citationCount"": 0, ""container"": {}, ""content..."
532,datacite::10.11588/propylaeumdok.00000147,PropylaeumDok,datacite,10.11588/propylaeumdok.00000147,10.11588/propylaeumdok.00000147,https://doi.org/10.11588/propylaeumdok.00000147,http://archiv.ub.uni-heidelberg.de/propylaeumd...,http://archiv.ub.uni-heidelberg.de/propylaeumd...,10.11588,None,gesis.ubhd,gesis,datacite,Heidelberg University Library,None,None,None,None,OCCIDENT & ORIENT: Newsletter of the German Pr...,None,None,None,None,Collection,Periodical,Collection,None,None,2017-02-23,None,None,None,2020-12-10,None,2017-02-23,None,None,2008.0,published_year,None,None,None,None,None,[],None,None,None,Deutsches Evangelisches Institut Für Altertums...,None,None,"[{""affiliation"": [], ""name"": ""Deutsches Evange...",[],None,[],None,None,"[{""subject"": ""Palä

# PsyArXiv

In [79]:
PsyArXiv_df, PsyArXiv_summary = get_server_data("PsyArXiv")


 SERVER ANALYSIS: PSYARXIV
  > Files found:    1
  > Raw records:    56866
OK


In [80]:
PsyArXiv_parent_df, PsyArXiv_parent_summary = analyze_parent_data("PsyArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: PSYARXIV
  > Total Parent Groups:     48,701
  > Total Versions Found:    59,523
  > Unique Parent DOIs:      48,701
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for PsyArXiv:
-----------------------------------
  1 version(s):     40,523 groups
  2 version(s):     6,500 groups
  3 version(s):     1,243 groups
  4 version(s):     255 groups
  5 version(s):     99 groups
  6 version(s):     36 groups
  7 version(s):     16 groups
  8 version(s):     7 groups
  9 version(s):     3 groups
  10 version(s):    4 groups
  12 version(s):    4 groups
  14 version(s):    1 groups
  15 version(s):    1 groups
  16 version(s):    1 groups
  17 version(s):    1 groups
  18 version(s):    2 groups
  21 version(s):    1 groups
  24 version(s):    1 groups
  25 version(s):    1 groups
  37 version(s):    1 groups
  52 version(s):    1 groups

  Average versions per parent: 1.22

 MOST RECENT DESTINATIONS (Migration):
------------

# Qeios

In [81]:
Qeios_df, Qeios_summary = get_server_data("Qeios")


 SERVER ANALYSIS: QEIOS
  > Files found:    1
  > Raw records:    5650
OK


In [82]:
Qeios_parent_df, Qeios_parent_summary = analyze_parent_data("Qeios", full_parent_df)


 CONSOLIDATED ANALYSIS: QEIOS
  > Total Parent Groups:     3,256
  > Total Versions Found:    5,135
  > Unique Parent DOIs:      3,256
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Qeios:
-----------------------------------
  1 version(s):     2,084 groups
  2 version(s):     829 groups
  3 version(s):     210 groups
  4 version(s):     54 groups
  5 version(s):     37 groups
  6 version(s):     12 groups
  7 version(s):     10 groups
  8 version(s):     4 groups
  9 version(s):     4 groups
  11 version(s):    3 groups
  12 version(s):    3 groups
  13 version(s):    3 groups
  14 version(s):    1 groups
  15 version(s):    2 groups

  Average versions per parent: 1.58

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Qeios                     3190
SSRN                        19
arXiv                        9
ResearchGate                 7
Research Square              6
Qeios; Rese

# RePEc: Research Papers in Economics

In [83]:
# RePEc_df, RePEc_summary = get_server_data("RePEc_Research_Papers_in_Economics")

In [84]:
RePEc_parent_df, RePEc_parent_summary = analyze_parent_data("RePEc: Research Papers in Economics", full_parent_df)


 CONSOLIDATED ANALYSIS: REPEC: RESEARCH PAPERS IN ECONOMICS
  > Total Parent Groups:     369,997
  > Total Versions Found:    430,371
  > Unique Parent DOIs:      20,673
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for RePEc: Research Papers in Economics:
-----------------------------------
  1 version(s):     315,721 groups
  2 version(s):     49,392 groups
  3 version(s):     4,096 groups
  4 version(s):     611 groups
  5 version(s):     116 groups
  6 version(s):     25 groups
  7 version(s):     12 groups
  8 version(s):     6 groups
  9 version(s):     4 groups
  10 version(s):    4 groups
  11 version(s):    4 groups
  14 version(s):    1 groups
  18 version(s):    1 groups
  22 version(s):    1 groups
  27 version(s):    1 groups
  28 version(s):    1 groups
  43 version(s):    1 groups

  Average versions per parent: 1.16

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
RePEc

# Research Square

In [85]:
# Research_Square_df, Research_Square_summary = get_server_data("Research_Square")

In [86]:
Research_Square_parent_df, Research_Square_parent_summary = analyze_parent_data("Research Square", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCH SQUARE
  > Total Parent Groups:     389,983
  > Total Versions Found:    439,160
  > Unique Parent DOIs:      389,983
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Research Square:
-----------------------------------
  1 version(s):     355,085 groups
  2 version(s):     25,294 groups
  3 version(s):     6,449 groups
  4 version(s):     2,253 groups
  5 version(s):     610 groups
  6 version(s):     174 groups
  7 version(s):     66 groups
  8 version(s):     24 groups
  9 version(s):     11 groups
  10 version(s):    5 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    1 groups
  14 version(s):    1 groups
  17 version(s):    1 groups
  21 version(s):    1 groups
  24 version(s):    2 groups
  25 version(s):    1 groups
  47 version(s):    1 groups

  Average versions per parent: 1.13

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_rece

# ResearchGate

In [87]:
ResearchGate_df, ResearchGate_summary = get_server_data("ResearchGate")


 SERVER ANALYSIS: RESEARCHGATE
  > Files found:    1
  > Raw records:    181231
OK


In [88]:
ResearchGate_parent_df, ResearchGate_parent_summary = analyze_parent_data("ResearchGate", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCHGATE
  > Total Parent Groups:     167,312
  > Total Versions Found:    180,737
  > Unique Parent DOIs:      167,312
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ResearchGate:
-----------------------------------
  1 version(s):     157,458 groups
  2 version(s):     8,116 groups
  3 version(s):     1,231 groups
  4 version(s):     241 groups
  5 version(s):     97 groups
  6 version(s):     43 groups
  7 version(s):     44 groups
  8 version(s):     16 groups
  9 version(s):     13 groups
  10 version(s):    9 groups
  11 version(s):    7 groups
  12 version(s):    4 groups
  13 version(s):    5 groups
  14 version(s):    1 groups
  15 version(s):    2 groups
  17 version(s):    4 groups
  18 version(s):    2 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    3 groups
  22 version(s):    1 groups
  23 version(s):    1 groups
  25 version(s):    1 groups
  27 version(s): 

# ResearchHub

In [89]:
ResearchHub_df, ResearchHub_summary = get_server_data("ResearchHub")


 SERVER ANALYSIS: RESEARCHHUB
  > Files found:    1
  > Raw records:    1636
OK


In [90]:
ResearchHub_parent_df, ResearchHub_parent_summary = analyze_parent_data("ResearchHub", full_parent_df)


 CONSOLIDATED ANALYSIS: RESEARCHHUB
  > Total Parent Groups:     1,375
  > Total Versions Found:    1,619
  > Unique Parent DOIs:      1,375
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ResearchHub:
-----------------------------------
  1 version(s):     1,260 groups
  2 version(s):     71 groups
  3 version(s):     17 groups
  4 version(s):     8 groups
  5 version(s):     4 groups
  6 version(s):     7 groups
  7 version(s):     3 groups
  8 version(s):     3 groups
  10 version(s):    1 groups
  17 version(s):    1 groups

  Average versions per parent: 1.18

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ResearchHub                  1356
ResearchHub; Zenodo             9
Zenodo                          4
Authorea Inc.                   2
ResearchGate; ResearchHub       2
SocArXiv                        1
EarthArXiv                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOK

# SAE Mobilus®

In [91]:
SAE_df, SAE_summary = get_server_data("SAE_Mobilus®")


 SERVER ANALYSIS: SAE_MOBILUS®
  > Files found:    1
  > Raw records:    105
OK


In [92]:
SAE_parent_df, SAE_parent_summary = analyze_parent_data("SAE Mobilus®", full_parent_df)


 CONSOLIDATED ANALYSIS: SAE MOBILUS®
  > Total Parent Groups:     104
  > Total Versions Found:    108
  > Unique Parent DOIs:      104
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SAE Mobilus®:
-----------------------------------
  1 version(s):     100 groups
  2 version(s):     4 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SAE Mobilus®    101
arXiv             2
SSRN              1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.47953/sae-pp-    104



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
saemobilus.sae.org    104


 COMPLETED: SAE MOBILUS®



# SciELO Preprints

In [93]:
SciELO_df, SciELO_summary = get_server_data("SciELO_Preprints")


 SERVER ANALYSIS: SCIELO_PREPRINTS
  > Files found:    1
  > Raw records:    4141
OK


In [94]:
SciELO_parent_df, SciELO_parent_summary = analyze_parent_data("SciELO Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIELO PREPRINTS
  > Total Parent Groups:     4,084
  > Total Versions Found:    4,142
  > Unique Parent DOIs:      4,084
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SciELO Preprints:
-----------------------------------
  1 version(s):     4,038 groups
  2 version(s):     37 groups
  3 version(s):     7 groups
  4 version(s):     1 groups
  5 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SciELO Preprints                  4066
ResearchGate; SciELO Preprints       3
Research Square                      3
Zenodo                               2
medRxiv                              2
SciELO Preprints; Zenodo             2
Qeios; SciELO Preprints              1
SSRN                                 1
PsyArXiv                             1
Preprints.org                        1

 TOP VALUES FOR: DOI_

# ScienceOpen Preprints

In [95]:
ScienceOpen_df, ScienceOpen_summary = get_server_data("ScienceOpen_Preprints")


 SERVER ANALYSIS: SCIENCEOPEN_PREPRINTS
  > Files found:    1
  > Raw records:    2970
OK


In [96]:
ScienceOpen_df[ScienceOpen_df['subtype_backend_raw']=='other']

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
1,crossref::10.14293/s2199-1006.1.sor-.ppzm4he.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppzm4he.v1,10.14293/s2199-1006.1.sor-.ppzm4he.v1,https://doi.org/10.14293/s2199-1006.1.sor-.ppz...,https://scienceopen.com/document?vid=e8537577-...,https://scienceopen.com/document?vid=e8537577-...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Towards exploring adaptive and associatively r...,None,None,None,None,posted-content,other,other,None,True,2020-08-04,2019-01-10,2020-09-08,2025-05-14,None,2019-01-10,None,2019-01-10,None,2019,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<ns7:p>This poster presents an initial set of ...,This poster presents an initial set of observa...,"[{""URL"": ""https://scienceopen.com/document?vid...",None,"Olteteanu, Ana-Maria; Dyer, Jonathan",None,None,"[{""ORCID"": ""https://orcid.org/0000-0002-0639-7...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,1,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppzm4he.v1..."
6,crossref::10.14293/s2199-1006.1.sor-.ppsvskm.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.ppsvskm.v1,10.14293/s2199-1006.1.sor-.ppsvskm.v1,https://doi.org/10.14293/s2199-1006.1.sor-.pps...,https://scienceopen.com/hosted-document?doi=10...,https://scienceopen.com/hosted-document?doi=10...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,Stealth Adapted Coronaviruses Resulting from t...,None,None,None,None,posted-content,other,other,None,True,2021-05-10,2021-05-08,2023-03-17,2023-03-17,None,2021-05-08,None,2021-05-08,None,2021,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,<jats:p>The continuing emergence of variant fo...,The continuing emergence of variant forms of t...,"[{""URL"": ""https://scienceopen.com/hosted-docum...",None,"Martin, W John","Institute of Progressive Medicine, 1634 Spruce...",None,"[{""affiliation"": [{""name"": ""Institute of Progr...",None,None,None,None,NaN,None,None,None,0,None,None,0,0,None,None,,,False,None,,None,None,,None,NaN,None,doi_prefix_first_token/prefix,1,None,"{""DOI"": ""10.14293/s2199-1006.1.sor-.ppsvskm.v1..."
72,crossref::10.14293/s2199-1006.1.sor-.pptzsif.v1,ScienceOpen Preprints,crossref,10.14293/s2199-1006.1.sor-.pptzsif.v1,10.14293/s2199-1006.1.sor-.pptzsif.v1,https://doi.org/10.14293/s2199-1006.1.sor-.ppt...,https://scienceopen.com/hosted-document?doi=10...,https://scienceopen.com/hosted-document?doi=10...,10.14293,5403,None,None,crossref,ScienceOpen,None,ScienceOpen,None,None,MENTAL ILLNESS: AN INVISIBLE TRAUMA,None,None,None,None,posted-content,other,other,None,True,2021-06-22,2021-06-21,

In [97]:
ScienceOpen_parent_df, ScienceOpen_parent_summary = analyze_parent_data("ScienceOpen Preprints", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIENCEOPEN PREPRINTS
  > Total Parent Groups:     1,624
  > Total Versions Found:    2,027
  > Unique Parent DOIs:      1,624
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for ScienceOpen Preprints:
-----------------------------------
  1 version(s):     1,342 groups
  2 version(s):     207 groups
  3 version(s):     49 groups
  4 version(s):     16 groups
  5 version(s):     5 groups
  6 version(s):     2 groups
  7 version(s):     2 groups
  9 version(s):     1 groups

  Average versions per parent: 1.25

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
ScienceOpen Preprints                   1474
SSRN                                      33
Preprints.org                             17
Open Science Framework                    13
ResearchGate                              10
Authorea Inc.                             10
Research Square                           

# Sciencepaper Online

In [98]:
Sciencepaper_df, Sciencepaper_summary = get_server_data("Sciencepaper_Online")


 SERVER ANALYSIS: SCIENCEPAPER_ONLINE
  > Files found:    1
  > Raw records:    99
OK


In [99]:
Sciencepaper_parent_df, Sciencepaper_parent_summary = analyze_parent_data("Sciencepaper Online", full_parent_df)


 CONSOLIDATED ANALYSIS: SCIENCEPAPER ONLINE
  > Total Parent Groups:     97
  > Total Versions Found:    97
  > Unique Parent DOIs:      97
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Sciencepaper Online:
-----------------------------------
  1 version(s):     97 groups

  Average versions per parent: 1.00

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Sciencepaper Online    97

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.61951/sciencepaperonline    97



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
paper.edu.cn    97


 COMPLETED: SCIENCEPAPER ONLINE



# searchRxiv

In [100]:
searchRxiv_df, searchRxiv_summary = get_server_data("searchRxiv")


 SERVER ANALYSIS: SEARCHRXIV
  > Files found:    2
  > Raw records:    1234
OK


In [101]:
searchRxiv_parent_df, searchRxiv_parent_summary = analyze_parent_data("searchRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SEARCHRXIV
  > Total Parent Groups:     529
  > Total Versions Found:    1,221
  > Unique Parent DOIs:      529
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for searchRxiv:
-----------------------------------
  1 version(s):     368 groups
  2 version(s):     28 groups
  3 version(s):     26 groups
  4 version(s):     27 groups
  5 version(s):     20 groups
  6 version(s):     18 groups
  7 version(s):     16 groups
  8 version(s):     9 groups
  9 version(s):     6 groups
  10 version(s):    2 groups
  11 version(s):    4 groups
  14 version(s):    2 groups
  15 version(s):    1 groups
  21 version(s):    1 groups
  37 version(s):    1 groups

  Average versions per parent: 2.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
searchRxiv         518
Research Square      6
JMIR Preprints       3
PsyArXiv             1
Zenodo               1

 TOP VALUES FOR: DO

# SocArXiv

In [102]:
SocArXiv_df, SocArXiv_summary = get_server_data("SocArXiv")


 SERVER ANALYSIS: SOCARXIV
  > Files found:    1
  > Raw records:    21541
OK


In [103]:
SocArXiv_parent_df, SocArXiv_parent_summary = analyze_parent_data("SocArXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SOCARXIV
  > Total Parent Groups:     18,026
  > Total Versions Found:    23,686
  > Unique Parent DOIs:      18,026
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SocArXiv:
-----------------------------------
  1 version(s):     13,586 groups
  2 version(s):     3,439 groups
  3 version(s):     846 groups
  4 version(s):     117 groups
  5 version(s):     27 groups
  6 version(s):     4 groups
  7 version(s):     3 groups
  8 version(s):     2 groups
  10 version(s):    2 groups

  Average versions per parent: 1.31

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SocArXiv                            14741
SSRN                                 1108
Open Science Framework; SocArXiv     1026
Open Science Framework                868
CrimRxiv                               43
arXiv                                  38
SocArXiv; arXiv                        26
Resear

# Social Science Open Access Repository

In [104]:
Social_Science_Open_Access_Repository_df, Social_Science_Open_Access_Repository_summary = get_server_data("Social_Science_Open_Access_Repository")


 SERVER ANALYSIS: SOCIAL_SCIENCE_OPEN_ACCESS_REPOSITORY
  > Files found:    1
  > Raw records:    27201
OK


In [105]:
ssoar_parent_df, ssoar_parent_summary = analyze_parent_data("Social Science Open Access Repository", full_parent_df)


 CONSOLIDATED ANALYSIS: SOCIAL SCIENCE OPEN ACCESS REPOSITORY
  > Total Parent Groups:     26,905
  > Total Versions Found:    27,266
  > Unique Parent DOIs:      6,581
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Social Science Open Access Repository:
-----------------------------------
  1 version(s):     26,597 groups
  2 version(s):     274 groups
  3 version(s):     24 groups
  4 version(s):     2 groups
  5 version(s):     7 groups
  6 version(s):     1 groups

  Average versions per parent: 1.01

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Social Science Open Access Repository                                         26741
SSRN                                                                             80
RePEc: Research Papers in Economics                                              17
RePEc: Research Papers in Economics; Social Science Open Access Repository       13


# SportRxiv

In [106]:
SportRxiv_df, SportRxiv_summary = get_server_data("SportRxiv")


 SERVER ANALYSIS: SPORTRXIV
  > Files found:    2
  > Raw records:    878
OK


In [107]:
SportRxiv_parent_df, SportRxiv_parent_summary = analyze_parent_data("SportRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: SPORTRXIV
  > Total Parent Groups:     847
  > Total Versions Found:    898
  > Unique Parent DOIs:      847
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SportRxiv:
-----------------------------------
  1 version(s):     800 groups
  2 version(s):     43 groups
  3 version(s):     4 groups

  Average versions per parent: 1.06

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
SportRxiv                            810
Open Science Framework; SportRxiv     28
JMIR Preprints                         2
Research Square                        2
arXiv                                  2
Open Science Framework                 1
ResearchGate; SportRxiv                1
SSRN                                   1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.51224/srxiv    480
10.31236/osf      366
10.31236/srxiv      

# SSRN

In [ ]:
SSRN_df, SSRN_summary = get_server_data("SSRN")

In [109]:
SSRN_parent_df, SSRN_parent_summary = analyze_parent_data("SSRN", full_parent_df)


 CONSOLIDATED ANALYSIS: SSRN
  > Total Parent Groups:     1,120,944
  > Total Versions Found:    1,235,560
  > Unique Parent DOIs:      1,120,944
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for SSRN:
-----------------------------------
  1 version(s):     1,022,779 groups
  2 version(s):     85,658 groups
  3 version(s):     10,164 groups
  4 version(s):     1,749 groups
  5 version(s):     349 groups
  6 version(s):     104 groups
  7 version(s):     53 groups
  8 version(s):     29 groups
  9 version(s):     15 groups
  10 version(s):    8 groups
  11 version(s):    3 groups
  12 version(s):    4 groups
  13 version(s):    7 groups
  14 version(s):    3 groups
  15 version(s):    1 groups
  16 version(s):    4 groups
  17 version(s):    1 groups
  19 version(s):    1 groups
  20 version(s):    1 groups
  21 version(s):    1 groups
  22 version(s):    2 groups
  24 version(s):    2 groups
  28 version(s):    1 groups
  30 version(s):   

# TechRxiv

In [110]:
TechRxiv_df, TechRxiv_summary = get_server_data("TechRxiv")


 SERVER ANALYSIS: TECHRXIV
  > Files found:    1
  > Raw records:    29418
OK


In [111]:
TechRxiv_parent_df, TechRxiv_parent_summary = analyze_parent_data("TechRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: TECHRXIV
  > Total Parent Groups:     16,216
  > Total Versions Found:    27,311
  > Unique Parent DOIs:      16,216
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for TechRxiv:
-----------------------------------
  1 version(s):     7,677 groups
  2 version(s):     6,764 groups
  3 version(s):     1,291 groups
  4 version(s):     336 groups
  5 version(s):     79 groups
  6 version(s):     38 groups
  7 version(s):     9 groups
  8 version(s):     11 groups
  9 version(s):     4 groups
  10 version(s):    2 groups
  11 version(s):    2 groups
  12 version(s):    2 groups
  13 version(s):    1 groups

  Average versions per parent: 1.68

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
TechRxiv                  15748
arXiv                       209
SSRN                         93
Research Square              48
TechRxiv; arXiv              39
Authorea Inc.        

# Therapoid

In [112]:
Therapoid_df, Therapoid_summary = get_server_data("Therapoid")


 SERVER ANALYSIS: THERAPOID
  > Files found:    1
  > Raw records:    7
OK


In [113]:
Therapoid_parent_df, Therapoid_parent_summary = analyze_parent_data("Therapoid", full_parent_df)


 CONSOLIDATED ANALYSIS: THERAPOID
  > Total Parent Groups:     6
  > Total Versions Found:    7
  > Unique Parent DOIs:      6
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Therapoid:
-----------------------------------
  1 version(s):     5 groups
  2 version(s):     1 groups

  Average versions per parent: 1.17

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Therapoid    6

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.24973/20    6



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
therapoid.net    6


 COMPLETED: THERAPOID



# Thesis Commons

In [114]:
Thesis_df, Thesis_summary = get_server_data("Thesis_Commons")


 SERVER ANALYSIS: THESIS_COMMONS
  > Files found:    1
  > Raw records:    3959
OK


In [115]:
Thesis_parent_df, Thesis_parent_summary = analyze_parent_data("Thesis Commons", full_parent_df)


 CONSOLIDATED ANALYSIS: THESIS COMMONS
  > Total Parent Groups:     3,220
  > Total Versions Found:    4,327
  > Unique Parent DOIs:      3,220
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Thesis Commons:
-----------------------------------
  1 version(s):     2,734 groups
  2 version(s):     393 groups
  3 version(s):     57 groups
  4 version(s):     14 groups
  5 version(s):     5 groups
  6 version(s):     1 groups
  7 version(s):     3 groups
  8 version(s):     2 groups
  9 version(s):     1 groups
  11 version(s):    1 groups
  13 version(s):    1 groups
  14 version(s):    1 groups
  16 version(s):    2 groups
  29 version(s):    1 groups
  46 version(s):    1 groups
  100 version(s):   1 groups
  108 version(s):   1 groups
  150 version(s):   1 groups

  Average versions per parent: 1.34

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Thesis Commons                      

# UCL Open Environment

In [116]:
UCL_df, UCL_summary = get_server_data("UCL_Open_Environment")


 SERVER ANALYSIS: UCL_OPEN_ENVIRONMENT
  > Files found:    2
  > Raw records:    369
OK


/tmp/ipykernel_3588/1436864115.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)


In [117]:
UCL_parent_df, UCL_parent_summary = analyze_parent_data("UCL Open Environment", full_parent_df)


 CONSOLIDATED ANALYSIS: UCL OPEN ENVIRONMENT
  > Total Parent Groups:     173
  > Total Versions Found:    366
  > Unique Parent DOIs:      173
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for UCL Open Environment:
-----------------------------------
  1 version(s):     85 groups
  2 version(s):     15 groups
  3 version(s):     48 groups
  4 version(s):     22 groups
  5 version(s):     2 groups
  9 version(s):     1 groups

  Average versions per parent: 2.12

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
UCL Open Environment    173

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.14324/11                150
10.14324/ucloepreprints     23



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
journals.uclpress.co.uk    171
ucl.scienceopen.com          2


 COMPLETED: UCL OPEN ENVIRONMENT



# UnisaRxiv

In [118]:
UnisaRxiv_df, UnisaRxiv_summary = get_server_data("UnisaRxiv")


 SERVER ANALYSIS: UNISARXIV
  > Files found:    1
  > Raw records:    126
OK


In [119]:
UnisaRxiv_parent_df, UnisaRxiv_parent_summary = analyze_parent_data("UnisaRxiv", full_parent_df)


 CONSOLIDATED ANALYSIS: UNISARXIV
  > Total Parent Groups:     120
  > Total Versions Found:    125
  > Unique Parent DOIs:      120
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for UnisaRxiv:
-----------------------------------
  1 version(s):     116 groups
  2 version(s):     3 groups
  3 version(s):     1 groups

  Average versions per parent: 1.04

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
UnisaRxiv       117
SSRN              1
ResearchGate      1
AfricArXiv        1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.25159/unisarxiv    120



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
scienceopen.com    120


 COMPLETED: UNISARXIV



# VeriXiv

In [120]:
VeriXiv_df, VeriXiv_summary = get_server_data("VeriXiv")


 SERVER ANALYSIS: VERIXIV
  > Files found:    1
  > Raw records:    504
OK


In [121]:
VeriXiv_parent_df, VeriXiv_parent_summary = analyze_parent_data("VeriXiv", full_parent_df)


 CONSOLIDATED ANALYSIS: VERIXIV
  > Total Parent Groups:     425
  > Total Versions Found:    507
  > Unique Parent DOIs:      425
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for VeriXiv:
-----------------------------------
  1 version(s):     368 groups
  2 version(s):     38 groups
  3 version(s):     13 groups
  4 version(s):     6 groups

  Average versions per parent: 1.19

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
VeriXiv                399
Gates Open Research     24
AgEcon Search            1
Research Square          1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/verixiv    425



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
verixiv.org    425


 COMPLETED: VERIXIV



In [122]:
VeriXiv_parent_df[VeriXiv_parent_df['most_recent_version_servers']=='Research Square']

,dup_group_id_full,parent_record_id,parent_server_name,parent_doi,parent_url,parent_title,parent_authors,parent_date_first_seen,parent_year_first_seen,version_record_ids,version_dois,servers_with_counts,total_versions,first_version_date,last_version_date,most_recent_version_servers,doi_lc,prefix_lc,doi_prefix_from_text,doi_suffix,doi_prefix_first_token,doi_prefix_bucket_2d,primary_domain,primary_domain_extend
7125739,crossref::10.12688/verixiv.2010.1,crossref::10.12688/verixiv.2010.1,VeriXiv,10.12688/verixiv.2010.1,https://verixiv.org/articles/2-286/v1,Maternal screening coverage and determinants d...,"Adelabu, Yusuf; Saalu, Tersur T.; Afolabi, Bos...",2025-09-15,2025,crossref::10.12688/verixiv.2010.1; crossref::1...,10.12688/verixiv.2010.1; 10.21203/rs.3.rs-7490...,Research Square (1); VeriXiv (1),2,2025-09-15,2025-09-17,Research Square,10.12688/verixiv.2010.1,10.12688,10.12688,verixiv.2010.1,10.12688/verixiv,10.12688/verixiv,verixiv.org,verixiv.org/articles


In [123]:
VeriXiv_df

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
0,crossref::10.12688/verixiv.83.1,VeriXiv,crossref,10.12688/verixiv.83.1,10.12688/verixiv.83.1,https://doi.org/10.12688/verixiv.83.1,https://verixiv.org/articles/1-14/v1,https://verixiv.org/articles/1-14/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates Foundation,None,Developing a male-specific counselling curricu...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-10,2024-10-10,2024-10-11,2025-11-21,None,2024-10-10,None,2024-10-10,None,2024,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Mphande, Misheck; Robson, Isabella; Hubbard, J...",None,None,"[{""affiliation"": [], ""family"": ""Mphande"", ""giv...",None,None,"[{""DOI"": ""10.13039/100000061"", ""award"": [""K01-...",Fogarty International Center; Bill and Melinda...,3.0,None,None,None,0,None,None,0,63,"[{""DOI"": ""10.1080/17441690902942464"", ""article...",None,,,False,None,,None,None,,None,None,None,prefix/primary_domain,40,None,"{""DOI"": ""10.12688/verixiv.83.1"", ""URL"": ""https..."
1,crossref::10.12688/verixiv.197.1,VeriXiv,crossref,10.12688/verixiv.197.1,10.12688/verixiv.197.1,https://doi.org/10.12688/verixiv.197.1,https://verixiv.org/articles/1-15/v1,https://verixiv.org/articles/1-15/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates Foundation,None,Discovery of a picomolar antiplasmodial pyrazo...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-10-16,2024-10-16,2024-10-18,2025-07-23,None,2024-10-16,None,2024-10-16,None,2024,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,None,None,None,None,"Tchatat Tali, Mariscal Brice; Dize, Darline; Y...",None,None,"[{""affiliation"": [], ""family"": ""Tchatat Tali"",...",None,None,"[{""DOI"": ""10.13039/100000865"", ""award"": [""INV-...",Bill and Melinda Gates Foundation,1.0,None,None,None,1,None,None,1,65,"[{""DOI"": ""10.1021/jm990002y"", ""article-title"":...",None,,,False,None,,None,None,,None,None,None,prefix/primary_domain,40,None,"{""DOI"": ""10.12688/verixiv.197.1"", ""URL"": ""http..."
2,crossref::10.12688/verixiv.77.1,VeriXiv,crossref,10.12688/verixiv.77.1,10.12688/verixiv.77.1,https://doi.org/10.12688/verixiv.77.1,https://verixiv.org/articles/1-9/v1,https://verixiv.org/articles/1-9/v1,10.12688,2560,None,None,crossref,F1000 Research Ltd,None,None,Gates Foundation,None,An exploration of unusual antimicrobial resist...,None,None,None,None,posted-content,preprint,preprint,None,True,2024-09-18,2024-09-17,2024-10-21,2025-11-21,None,2024-09-17,None,2024-09-17,None,2024,issued_date,posted_date,None,None,http://creativecommons.org/licenses/by/4.0/,http://creativecommons.org/licenses/by/4.0/,None,None,"[{""URL"": ""https://ve

In [124]:
VeriXiv_df['funders_flat'].value_counts()

funders_flat
Bill and Melinda Gates Foundation                                                                                                                                                                                                                                                          261
Gates Foundation                                                                                                                                                                                                                                                                            93
Bill and Melinda Gates Foundation; African Economic Research Consortium                                                                                                                                                                                                                      4
Bill and Melinda Gates Foundation; Children’s Investment Fund Foundation                                                      

In [125]:
# Optional: Group similar names together
df_funders = VeriXiv_df['funders_flat'].dropna().str.split(';').explode().str.strip()

# # Normalize common variants
# df_funders = df_funders.replace({
#     "Gates Foundation": "Bill and Melinda Gates Foundation"
# })

final_counts = df_funders.value_counts().to_frame()
final_counts.head(10)

,count
funders_flat,
Bill and Melinda Gates Foundation,366
Gates Foundation,120
Wellcome Trust,9
Biotechnology and Biological Sciences Research Council,8
Children's Investment Fund Foundation,7
National Institute of Allergy and Infectious Diseases,7
European and Developing Countries Clinical Trials Partnership,6
Consortium pour la recherche économique en Afrique,6
Wellcome,6


# viXra

In [126]:
viXra_df, viXra_summary = get_server_data("viXra")


 SERVER ANALYSIS: VIXRA
  > Files found:    1
  > Raw records:    25570
OK


In [127]:
viXra_parent_df, viXra_parent_summary = analyze_parent_data("viXra", full_parent_df)


 CONSOLIDATED ANALYSIS: VIXRA
  > Total Parent Groups:     22,570
  > Total Versions Found:    23,011
  > Unique Parent DOIs:      0
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for viXra:
-----------------------------------
  1 version(s):     22,228 groups
  2 version(s):     287 groups
  3 version(s):     37 groups
  4 version(s):     8 groups
  5 version(s):     3 groups
  6 version(s):     3 groups
  7 version(s):     3 groups
  12 version(s):    1 groups

  Average versions per parent: 1.02

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
viXra                     22382
arXiv                        85
ResearchGate                 57
SSRN                         18
Zenodo                        8
HAL                           4
Open Science Framework        4
Preprints.org                 3
bioRxiv                       3
engrXiv                       1

 TOP VALUES FOR: DOI_PREF

# Wellcome Open Research

In [128]:
Wellcome_df, Wellcome_summary = get_server_data("Wellcome_Open_Research")


 SERVER ANALYSIS: WELLCOME_OPEN_RESEARCH
  > Files found:    1
  > Raw records:    4727
OK


In [129]:
Wellcome_parent_df, Wellcome_parent_summary = analyze_parent_data("Wellcome Open Research", full_parent_df)


 CONSOLIDATED ANALYSIS: WELLCOME OPEN RESEARCH
  > Total Parent Groups:     3,276
  > Total Versions Found:    4,480
  > Unique Parent DOIs:      3,276
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Wellcome Open Research:
-----------------------------------
  1 version(s):     2,264 groups
  2 version(s):     858 groups
  3 version(s):     122 groups
  4 version(s):     27 groups
  5 version(s):     4 groups
  6 version(s):     1 groups

  Average versions per parent: 1.37

 MOST RECENT DESTINATIONS (Migration):
-----------------------------------
most_recent_version_servers
Wellcome Open Research    3274
Authorea Inc.                1
medRxiv                      1

 TOP VALUES FOR: DOI_PREFIX_FIRST_TOKEN
-----------------------------------
doi_prefix_first_token
10.12688/wellcomeopenres    3276



 TOP VALUES FOR: PRIMARY_DOMAIN
-----------------------------------
primary_domain
wellcomeopenresearch.org    3276


 COMPLETED: WELLCOME

# Zenodo

In [6]:
Zenodo_df, Zenodo_summary = get_server_data("Zenodo")


 SERVER ANALYSIS: ZENODO
  > Files found:    1
  > Raw records:    166786
OK


In [7]:
record_ex = Zenodo_df[Zenodo_df['record_id']=='datacite::10.5281/zenodo.15050897']
record_ex

,record_id,server_name,backend,source_work_id,doi,doi_url,landing_page_url,url_best,prefix,member_id,client_id,provider_id,source_registry,publisher,container_title,institution_name,group_title,issn,title,original_title,short_title,subtitle,language,type_backend_raw,subtype_backend_raw,type_canonical,is_paratext,is_preprint_candidate,date_created,date_posted,date_deposited,date_indexed,date_updated,date_issued,date_registered,date_published,date_published_online,publication_year,date_published_source,date_posted_source,is_oa,oa_status,license,license_url_best,abstract_raw,abstract_text,links_json_best,fulltext_pdf_url,authors_flat,institutions_flat,countries_flat,authors_json,contributors_json,editors_json,funders_json,funders_flat,funders_count,subjects_json,concepts_json,topics_json,cited_by_count,cited_by_count_datacite,cited_by_count_openalex,is_referenced_by_count_crossref,reference_count,references_json,relations_json,has_preprint,is_preprint_of,has_published_version,published_version_ids_json,is_version_of,version_of_ids_json,version_label,has_review,update_to_json,parent_doi,update_policy,rule_tokens,rule_row_id,raw_relationships_json,raw_json
31120,datacite::10.5281/zenodo.15050897,Zenodo,datacite,10.5281/zenodo.15050897,10.5281/zenodo.15050897,https://doi.org/10.5281/zenodo.15050897,https://zenodo.org/doi/10.5281/zenodo.15050897,https://zenodo.org/doi/10.5281/zenodo.15050897,10.5281,None,cern.zenodo,cern,datacite,Zenodo,None,None,None,None,The Indeterminacy Space as the Foundation of S...,None,None,None,en,Preprint,,Preprint,None,True,2025-03-19,None,None,None,2025-08-25,None,2025-03-19,None,None,2025,published_year,None,None,None,Creative Commons Attribution 4.0 International,https://creativecommons.org/licenses/by/4.0/le...,"[{""description"": ""I present a new hypothesis i...",I present a new hypothesis in which \textbf{th...,None,None,"Paruzel, Katarzyna Anna",None,None,"[{""affiliation"": [], ""familyName"": ""Paruzel"", ...",[],None,[],None,NaN,"[{""subject"": ""theory of everything""}]",None,None,0,0,None,None,0,None,"[{""relatedIdentifier"": ""10.5281/zenodo.1505347...",,,False,None,10.5281/zenodo.15053474;10.5281/zenodo.1505352...,None,None,,None,None,None,client_id,10,"{""client"": {""data"": {""id"": ""cern.zenodo"", ""typ...","{""citationCount"": 0, ""container"": {}, ""content..."


In [10]:
record_ex['raw_json'][31120]

'{"citationCount": 0, "container": {}, "contentUrl": null, "contributors": [], "created": "2025-03-19T21:18:38Z", "creators": [{"affiliation": [], "familyName": "Paruzel", "givenName": "Katarzyna Anna", "name": "Paruzel, Katarzyna Anna", "nameIdentifiers": [{"nameIdentifier": "0009-0003-7818-9157", "nameIdentifierScheme": "ORCID"}], "nameType": "Personal"}], "dates": [{"date": "2025-02-26", "dateType": "Issued"}, {"date": "2025-03-19", "dateType": "Issued"}], "descriptions": [{"description": "I present a new hypothesis in which \\\\textbf{the indeterminacy space} constitutes the fundamental level of reality, from which spacetime, quantum mechanics, and gravity emergently arise.\\n\\nI introduce the field Φ as a mathematical description of the indeterminacy space—a stable potential state whose fluctuations, through a mechanism of spontaneous tunneling, lead to the emergence of classical spacetime geometry and generate quantum effects. In this framework, black holes are special regions w

In [131]:
Zenodo_parent_df, Zenodo_parent_summary = analyze_parent_data("Zenodo", full_parent_df)


 CONSOLIDATED ANALYSIS: ZENODO
  > Total Parent Groups:     67,908
  > Total Versions Found:    161,491
  > Unique Parent DOIs:      67,908
-----------------------------------------------------------------

 VERSIONING DISTRIBUTION for Zenodo:
-----------------------------------
  1 version(s):     6,942 groups
  2 version(s):     48,920 groups
  3 version(s):     6,665 groups
  4 version(s):     2,732 groups
  5 version(s):     942 groups
  6 version(s):     539 groups
  7 version(s):     290 groups
  8 version(s):     204 groups
  9 version(s):     122 groups
  10 version(s):    109 groups
  11 version(s):    71 groups
  12 version(s):    66 groups
  13 version(s):    35 groups
  14 version(s):    41 groups
  15 version(s):    18 groups
  16 version(s):    22 groups
  17 version(s):    20 groups
  18 version(s):    14 groups
  19 version(s):    12 groups
  20 version(s):    15 groups
  21 version(s):    5 groups
  22 version(s):    10 groups
  23 version(s):    11 groups
  24 versio